# COMP90055 Project

## System Design

In [1]:
import json
import pandas as pd
import numpy as np

# 1. Question Decomposition

In [2]:
### Read in Google Maps GeoQA Dataset ###

df_QA = pd.read_csv('../benchmark/benchmark_QA.csv')
df_QA

,Query,POI_Type,Origin,Origin_Latitude,Origin_Longitude,Origin_Address,Destination,Destination_Latitude,Destination_Longitude,Destination_Address,Waypoint,Waypoint_Latitude,Waypoint_Longitude,Waypoint_Address,Buffer_Range,Semantic_Requirements,Route,Answers
0,Find me an ATM with 24-hour service along the ...,atm,CERES Organic Cafe,-37.765672,144.982956,"Stewart St & Roberts St, Brunswick East VIC 30...",Melbourne TMJ & Facial Pain Centre,-37.872701,145.023372,"203 Balaclava Rd, Caulfield North VIC 3161, Au...",NaN,NaN,NaN,NaN,NaN,24-hour service,"[{""encodedPolyline"": ""~aoeFs|{sZHJ@F?\\PAjEf@b...","[{""727 T15 16/727 Collins St, Docklands VIC 30..."
1,Find me an ATM that is wheelchair accessible a...,atm,CERES Organic Cafe,-37.765672,144.982956,"Stewart St & Roberts St, Brunswick East VIC 30...",Melbourne TMJ & Facial Pain Centre,-37.872701,145.023372,"203 Balaclava Rd, Caulfield North VIC 3161, Au...",NaN,NaN,NaN,NaN,NaN,wheelchair accessible,"[{""encodedPolyline"": ""~aoeFs|{sZHJ@F?\\PAjEf@b...","[{""1 Scanlon Dr, Epping VIC 3076, Australia"": ..."
2,Find me a restaurant with a vegan menu along t...,restaurant,Romeo's of Toorak,-37.841244,145.009404,"450 Toorak Rd, Toorak VIC 3142, Australia",Fuso Port Melbourne,-37.825614,144.906248,"286-292 Lorimer St, Port Melbourne VIC 3207, A...",NaN,NaN,NaN,NaN,NaN,vegan menu,"[{""encodedPolyline"": ""dz}eF{catZ^oGmOwAK@a@?IB...","[{""1 Boathouse Rd, Kew VIC 3101, Australia"": {..."
3,Find me a restaurant rated 4 stars and above a...,restaurant,Romeo's of Toorak,-37.841244,145.009404,"450 Toorak Rd, Toorak VIC 3142, Australia",Fuso Port Melbourne,-37.825614,144.906248,"286-292 Lorimer St, Port Melbourne VIC 3207, A...",NaN,NaN,NaN,NaN,NaN,rated 4 stars,"[{""encodedPolyline"": ""dz}eF{catZ^oGmOwAK@a@?IB...","[{""27 Qv Square, Melbourne VIC 3000, Australia..."
4,Find me an amusement center with indoor play a...,amusement_center,Reservoir Leisure Centre,-37.716633,145.018923,"2A Cuthbert Rd, Reservoir VIC 3073, Australia","ALDI at 324/328 Hampshire Rd, Sunshine VIC 302...",-37.782752,144.834191,"324/328 Hampshire Rd, Sunshine VIC 3020, Austr...",NaN,NaN,NaN,NaN,NaN,indoor play areas,"[{""encodedPolyline"": ""~peeFc~btZ\\]TCH?{@xNFHA...","[{""330/340 McDonalds Rd, South Morang VIC 3752..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1925,Find me a park that allows dogs along the way ...,park,Prince Alfred Richmond,-37.832444,144.996936,"619 Church St, Richmond VIC 3121, Australia",KFC Coburg,-37.741116,144.962823,"137 Bell St, Coburg VIC 3058, Australia",NaN,NaN,NaN,NaN,NaN,allows dogs,"[{""encodedPolyline"": ""jc|eF}u~sZ\\qG~AN_@rH~JZ...","[{""Elliott Ave, Parkville VIC 3052, Australia""..."
1926,Find me a tourist attraction with guided tours...,tourist_attraction,The Pita Man,-37.873718,145.024739,"97 Hawthorn Rd, Caulfield North VIC 3161, Aust...",Calle Bakery,-37.789965,144.971542,"649 Rathdowne St, Carlton North VIC 3054, Aust...",NaN,NaN,NaN,NaN,NaN,guided tours,"[{""encodedPolyline"": ""pedfFgbdtZgOcBgHs@qRyBWK...","[{""180 St Kilda Rd, Melbourne VIC 3006, Austra..."
1927,Find me a tourist attraction that is family-fr...,tourist_attraction,The Pita Man,-37.873718,145.024739,"97 Hawthorn Rd, Caulfield North VIC 3161, Aust...",Calle Bakery,-37.789965,144.971542,"649 Rathdowne St, Carlton North VIC 3054, Aust...",NaN,NaN,NaN,NaN,NaN,family-friendly,"[{""encodedPolyline"": ""pedfFgbdtZgOcBgHs@qRyBWK...","[{""Queen St, Melbourne VIC 3000, Australia"": {..."
1928,Find me a gym that offers group classes along ...,gym,Spotlight Carnegie,-37.884420,145.059112,"Shop 14/2 Koornang Rd, Carnegie VIC 3163, Aust...",289 Asian Supermarket,-37.853280,145.159567,"287-293 Burwood Hwy, Burwood East VIC 3151, Au...",NaN,NaN,NaN,NaN,NaN,offers group classes,"[{""encodedPolyline"": ""nhffF}zjtZM@Wf@[zDa@Ev@{...","[{""29/41 Down St, Collingwood VIC 3066, Austra..."


In [7]:
query = df_QA[['Query']]

,Query
0,Find me an ATM with 24-hour service along the ...
1,Find me an ATM that is wheelchair accessible a...
2,Find me a restaurant with a vegan menu along t...
3,Find me a restaurant rated 4 stars and above a...
4,Find me an amusement center with indoor play a...
...,...
1925,Find me a park that allows dogs along the way ...
1926,Find me a tourist attraction with guided tours...
1927,Find me a tourist attraction that is family-fr...
1928,Find me a gym that offers group classes along ...


## 1.1 Prompting Style 1 -- Re-IR (IR: Intent Recognition)

Limitation: For more complex queries, LLMs might skip reasoning steps even with few-shot prompting

### version 1: Reason-while-Generation (RwG)

In [2]:
# ===================================
# Geo-Decompose Prompting Version 1.0
# ===================================

Geo_Decomp_system_prompt = """
You are a helpful assistant that helps breaking down complex geospatial queries into sub-queries in forms of JSON objects.
You will be given a complex geospatial query that requires both multi-hop geospatial reasoning and user semantic requirements analysis.

Please follow these instructions when constructing the sub-queries:

**Stage 1: Hierarchical (Top-down) Query Decomposition**
In this stage, you are going to break down the input geospatial query into into a nested JSON structure of intents and entities..

Please follow these three steps when breaking down the query:

    1. First, break down the query into multiple higher-level sub-queries. 
    2. Then, for each of the sub-query, generate appropriate reasoning thoughts to perform the Geospatial Intent Recognition.
    3. Finally, generate the corresponding JSON objects.
    
Please comply with the following rules when breaking down the query hierarchically:

    1. Do not return the final nested JSON object until no sub-query can be further decomposed.
    2. Do not generate the final results until the reasoning for all sub-queries finished.
    3. For each Geospatial Intent Recognition, try to first select one and only one "intent" strictly from this list: 
       ["computeRoutes", "searchNearby", "searchText", "searchAlongRoutes", "sampleRoutePoints", "geocodeLocation", "semanticComparison"]. If the intent cannot be described
       in any item within this list, you can construct your own intent.
    4. For each intent recognition, specify all the predicates that cannot be described by the "intent" object 
       into the "entity" object.
       
Explanation of the "intent" objects:
    - "computeRoutes": to find the path from origin to destination
    - "searchNearby": to find places or features within a specified radius around a point
    - "searchText": to search for places or features using a text query (e.g., “coffee shop Melbourne”)
    - "searchAlongRoutes": to find places or features located along a specified route between two points
    - "sampleRoutePoints": to generate or extract sample points along a route for further analysis (e.g., searchNearby)
    - "geocodeLocation": to get the geographical coordinates (latitude/longitude) of a place name or address
    - "semanticComparison": to compare two pieces of text or queries for meaning similarity (e.g., check if two queries refer to the similar user requirements)
**Reminder**: If you choose to use "searchAlongRoutes" intent, you cannot specify a buffer range in its "entity" object.

Please return a list of nested JSON objects as the final result for this stage.

**Stage 2: Flatten to Sequential Sub-queries**
In this stage, you are going to convert the nested structure from Stage 1 into an array of sub-queries with unique ids.

Please comply with the following rules when flattening the sub-queries.
    
    1. Each sub-query must have a unique "id". 
    2. Sub-query should refer to the results of earlier sub-queries using "$id" if required.
    3. Later sub-queries must refer to previous results with "$id" instead of nesting.
    4. Do not nest the full JSON of one sub-query inside another.
    5. List the most deeply nested sub-queries first, then proceed outward so later items can reference earlier ones.
    5. Remove any duplicated steps within the nested structure. 

--
## Examples

# Example 1

The geospatial query is: Find me a tourist attraction that has guided tours along the way from EG Ampol Coburg to Monash Medical Centre, within 6km of the route.

**Stage 1: Hierarchical (Top-down) Query Decomposition**
I need to find the tourist attraction from EG Ampol Coburg to Monash Medical Centre within 6km of the route.
I cannot intent to "searchAlongRoutes", otherwise I cannot apply the 6km buffer range.
An alternative intent is to "searchNearby" several route points along the route, applying the 6km buffer range.
Conducting Geospatial Intent Recognition to the query:
    - Intent: searchNearby (sub-query to find locations near the route points)
    - Entity: This would include the POI type (i.e., tourist_attraction), the reference location (route points), the buffer range (6000 meters), and the user semantic requirements (i.e. has guided tours)
Add the corresponding JSON object will be:

    {"intent": "searchNearby"
     "entity": <Route_points>, "POI_type": "tourist_attraction", "buffer_range": 6000, "user_requirements": "has guided tours"
    }
    
Inspecting the previous JSON object, the intent now is to derive <Route_points> (i.e. several points along the route from the origin to the destination).
Conducting Geospatial Intent Recognition to the query:
    - Intent: sampleRoutePoints (sub-query to sample multiple points along the route)
    - Entity: This would include the route along which points should be sampled (route), and the number of route points to sample (e.g., 2).
the expanded JSON object will be:

    {"intent": "searchNearby"
     "entity": {"Route_points": {"intent": "sampleRoutePoints",
                                 "entity": <Route>, "Num_route_points": 2
                                }
                "POI_type": "tourist_attraction",
                "buffer_range": 6000,
                "user_requirements": "has guided tours"
               }
    }
    
Inspecting the previous JSON object, the intent now is to derive the <Route> (i.e., the route from the origin to the destination).
Conducting Geospatial Intent Recognition to the query:
    - Intent: computeRoutes (to find the path from origin to destination)
    - Entity: This would involves the origin, and the destination.
the expanded JSON object will be:

    {"intent": "searchNearby"
     "entity": {"Route_points": {"intent": "sampleRoutePoints",
                                 "entity": {"Route": {"intent": "computeRoutes",
                                                      "entity": <Origin>, <Destination>
                                                      }
                                            }, 
                                            "num_route_points": 2
                                 }
                "POI_type": "tourist_attraction",
                "buffer_range": 6000,
                "user_requirements": "has guided tours"
               }
    }
    
Inspecting the previous JSON object, the intent now is to derive the <Origin> and <Destination> of the route.
Conducting Geospatial Intent Recognition to the query:
    - Intent: geocodeLocation (to get the geographical location of a place)
    - Entity: This would involves the origin (EG Ampol Coburg), and the destination (Monash Medical Centre).
the expanded JSON object will be:

    {"intent": "searchNearby"
         "entity": {"Route_points": {"intent": "sampleRoutePoints",
                                     "entity": {"Route": {"intent": "computeRoutes",
                                                          "entity": "Origin": {"intent": "geocodeLocation",
                                                                               "entity": "EG Ampol Coburg"}, 
                                                                    "Destination": {"intent": "geocodeLocation",
                                                                                    "entity": "Monash Medical Centre"}
                                                          }
                                                }, 
                                                "num_route_points": 2
                                     }
                    "POI_type": "tourist_attraction",
                    "buffer_range": 6000,
                    "user_requirements": "has guided tours"
                   }
    }

**Stage 2: Flatten to Sequential Sub-queries**
The decomposition of the geospatial query has been finished, we get a nested JSON object. Now, flatten the nested JSON object inside-out to get a linked JSON object.
{
  "sub_queries": [
    {
      "id": "q1",
      "intent": "geocodeLocation",
      "entity": {
        "place": "EG Ampol Coburg"
      }
    },
    {
      "id": "q2",
      "intent": "geocodeLocation",
      "entity": {
        "place": "Monash Medical Centre"
      }
    },
    {
      "id": "q3",
      "intent": "computeRoutes",
      "entity": {
        "origin": "$q1",
        "destination": "$q2"
      }
    },
    {
      "id": "q4",
      "intent": "sampleRoutePoints",
      "entity": {
        "route": "$q3",
        "num_route_points": 2
      }
    },
    {
      "id": "q5",
      "intent": "searchNearby",
      "entity": {
        "reference_location": "$q4",
        "POI_type": "tourist_attraction",
        "buffer_range": 6000,
        "user_requirements": "has guided tours"
      }
    }
  ]
}



### Example 2

The geospatial query is: Find me a fitness center with a swimming pool along the way from Bunnings Box Hill to Eleven Tigers.

**Stage 1: Hierarchical (Top-down) Query Decomposition**
I need to find a fitness center from Bunnings Box Hill to Eleven Tigers.
I can intent to "searchAlongRoutes", since the query does not specify a buffer range.
Conducting Geospatial Intent Recognition to the query:
    - Intent: searchAlongRoutes (sub-query to find places located along a specified route between two points)
    - Entity: This would include the POI type (i.e., fitness_center), the reference route (route), and the user semantic requirements (i.e. with a swimming pool)
the corresponding JSON object will be:

    {"intent": "searchAlongRoutes"
     "entity": {<Route>, "POI_type": "fitness_center", "user_requirements": "with a swimming pool"}
    }
    
    
Inspecting the previous JSON object, the intent now is to derive the <Route> (i.e., the route from the origin to the destination).
Conducting Geospatial Intent Recognition to the query:
    - Intent: computeRoutes (to find the path from origin to destination)
    - Entity: This would involves the origin (Bunnings Box Hill), and the destination (Eleven Tigers).
the expanded JSON object will be:

    {"intent": "searchAlongRoutes"
     "entity": {"Route": {"intent": "computeRoutes",
                          "entity": <Origin>, <Destination>
                          }, 
                "POI_type": "fitness_center", 
                "user_requirements": "with a swimming pool"
                }
    }
    
Inspecting the previous JSON object, the intent now is to derive the <Origin> and <Destination> of the route.
Conducting Geospatial Intent Recognition to the query:
    - Intent: geocodeLocation (to get the geographical location of a place)
    - Entity: This would involves the origin (Bunnings Box Hill), and the destination (Eleven Tigers).
the expanded JSON object will be:

    {"intent": "searchAlongRoutes"
     "entity": {"Route": {"intent": "computeRoutes",
                          "entity": "Origin": {"intent": "geocodeLocation",
                                               "entity": "Bunnings Box Hill"}, 
                                    "Destination": {"intent": "geocodeLocation",
                                                    "entity": "Eleven Tigers"}
                          }
                "POI_type": "fitness_center", 
                "user_requirements": "with a swimming pool"
               }
    }

**Stage 2: Flatten to Sequential Sub-queries**
The decomposition of the geospatial query has been finished, we get a nested JSON object. Now, flatten the nested JSON object inside-out to get a linked JSON object.

{
  "sub_queries": [
    {
      "id": "q1",
      "intent": "geocodeLocation",
      "entity": {
        "place": "Bunnings Box Hill"
      }
    },
    {
      "id": "q2",
      "intent": "geocodeLocation",
      "entity": {
        "place": "Eleven Tigers"
      }
    },
    {
      "id": "q3",
      "intent": "computeRoutes",
      "entity": {
        "origin": "$q1",
        "destination": "$q2"
      }
    },
    {
      "id": "q4",
      "intent": "searchAlongRoutes",
      "entity": {
        "route": "$q3",
        "POI_type": "fitness_center", 
        "user_requirements": "with a swimming pool"
      }
    }
  ]
}

"""


In [ ]:
# ===================================
# Prompting Template (version 1.0)
# ===================================

"""
You are a helpful assistant that helps breaking down complex geospatial queries into sub-queries in forms of JSON objects.
You will be given a complex geospatial query that requires both multi-hop geospatial reasoning and user semantic requirements analysis.
In order to structure the sub-queries systematically, you will be asked to perform a specific type of intent recognition called: Geospatial Intent Recognition.
You will be given a few examples showing how the Geospatial Intent Recognition works, try to follow and flexibly adapt to the structure of the example.

Please follow these three steps when breaking down the query:

    1. First, break down the query into multiple sub-queries. 
    2. Then, for each of the sub-query, generate appropriate reasoning thoughts to perform the Geospatial Intent Recognition.
    3. Finally, generate the corresponding JSON objects.
    
Please comply the following rules when breaking down the query:

    1. Do not return the final JSON object until no sub-qustion can be further decomposed.
    2. Do not generate the final results until the reasoning for all sub-queries finished.
    3. For each Geospatial Intent Recognition, try to first select one and only one "intent" strictly from this list: 
       ["computeRoutes", "searchNearby", "searchText", "searchAlongRoutes", "sampleRoutePoints", "geocodeLocation", "semanticComparison"]. If the intent cannot be described
       in any item within this list, you can construct your own intent.
    4. For each intent recognition, specify all the predicates that cannot be described by the "intent" object 
       into the "entity" object.
 
Please return a list of nested JSON objects as the final result.

Here is an example: 

The geospatial query is: Find me a tourist attraction that has guided tours along the way from EG Ampol Coburg to Monash Medical Centre, within 6km of the route.

I need to find the tourist attraction from EG Ampol Coburg to Monash Medical Centre, so I need to search along the route from origin (i.e. EG Ampol Coburg) to destination (i.e. Monash Medical Centre).
Conducting intent recognition to the query, the corresponding JSON object will be:

    {"intent": "search_along_route"
     "entity": <Route>, <POI_type>, <Buffer_range>
    }
    
Inspecting the previous JSON object, the task now is to derive <Route> (i.e. the route from the origin to the destination), <POI_TYPE> (i.e. places types) and <BUFFER_RANGE> (i.e. distance around the route) for the 'search_along_route' intent.
Conducting intent recognition to the query, the expanded JSON object will be:

    {"intent": "search_along_route"
     "entity": {"Route": {"intent": "compute_route",
                          "entity": <Origin>, <Destination>
                          }
                "POI_type": "tourist_attraction",
                "Buffer_range": 6000
               }
    }
    
Inspecting the previous JSON object, the task now is to derive the <Origin> and <Destination> for the 'compute_route' intent.
Conducting intent recognition to the query, the expanded JSON object will be:

    {"intent": "search_along_route"
     "entity": {"Route": {"intent": "compute_route",
                          "entity": "Origin": {"intent": "geo_code_location",
                                               "entity": "EG Ampol Coburg"}, 
                                    "Destination": {"intent": "geo_code_location",
                                                    "entity": "Monash Medical Centre"}
                          }
                "POI_type": "tourist_attraction",
                "Buffer_range": 6000
               }
    }
"""

This one works good, but have duplication logic within the query.

In [ ]:
# ===================================
# Prompting Template (version 1.0 final) 
# ===================================

"""
You are a helpful assistant that helps breaking down complex geospatial queries into sub-queries in forms of JSON objects.
You will be given a complex geospatial query that requires both multi-hop geospatial reasoning and user semantic requirements analysis.
In order to structure the sub-queries systematically, you will be asked to perform a specific type of intent recognition called: Geospatial Intent Recognition.
You will be given a few examples showing how the Geospatial Intent Recognition works, try to follow and flexibly adapt to the structure of the example.

Please follow these three steps when breaking down the query:

    1. First, break down the query into multiple sub-queries. 
    2. Then, for each of the sub-query, generate appropriate reasoning thoughts to perform the Geospatial Intent Recognition.
    3. Finally, generate the corresponding JSON objects.
    
Please comply the following rules when breaking down the query:

    1. Do not return the final JSON object until no sub-qustion can be further decomposed.
    2. Do not generate the final results until the reasoning for all sub-queries finished.
    3. For each Geospatial Intent Recognition, try to first select one and only one "intent" strictly from this list: 
       ["computeRoutes", "searchNearby", "searchText", "searchAlongRoutes", "sampleRoutePoints", "geocodeLocation", "semanticComparison"]. If the intent cannot be described
       in any item within this list, you can construct your own intent.
    4. For each intent recognition, specify all the predicates that cannot be described by the "intent" object 
       into the "entity" object.
       
Explanation of the "intent" objects:
    - "computeRoutes": to find the path from origin to destination
    - "searchNearby": to find places or features within a specified radius around a point
    - "searchText": to search for places or features using a text query (e.g., “coffee shop Melbourne”)
    - "searchAlongRoutes": to find places or features located along a specified route between two points
    - "sampleRoutePoints": to generate or extract sample points along a route for further analysis (e.g., searchNearby)
    - "geocodeLocation": to get the geographical coordinates (latitude/longitude) of a place name or address
    - "semanticComparison": to compare two pieces of text or queries for meaning similarity (e.g., check if two queries refer to the similar user requirements)
**Reminder**: If you choose to use "searchAlongRoutes" intent, you cannot specify a buffer range in its "entity" object.

Please return a list of nested JSON objects as the final result.

--
## Examples:

### Example 1

The geospatial query is: Find me a tourist attraction that has guided tours along the way from EG Ampol Coburg to Monash Medical Centre, within 6km of the route.

I need to find the tourist attraction from EG Ampol Coburg to Monash Medical Centre within 6km of the route.
I cannot intent to "searchAlongRoutes", otherwise I cannot apply the 6km buffer range.
An alternative intent is to "searchNearby" several route points along the route, applying the 6km buffer range.
Conducting intent recognition to the query:
    - Intent: searchNearby (sub-query to find locations near the route points)
    - Entity: This would include the POI type (i.e., tourist_attraction), the reference location (route points), the buffer range (6000 meters), and the user semantic requirements (i.e. has guided tours)
the corresponding JSON object will be:

    {"intent": "searchNearby"
     "entity": <Route_points>, "POI_type": "tourist_attraction", "Buffer_range": 6000, "User_requirements": "has guided tours"
    }
    
Inspecting the previous JSON object, the intent now is to derive <Route_points> (i.e. several points along the route from the origin to the destination).
Conducting intent recognition to the query:
    - Intent: sampleRoutePoints (sub-query to sample multiple points along the route)
    - Entity: This would include the route along which points should be sampled (route), and the number of route points to sample (e.g., 2).
the expanded JSON object will be:

    {"intent": "searchNearby"
     "entity": {"Route_points": {"intent": "sampleRoutePoints",
                                 "entity": <Route>, "Num_route_points": 2
                                }
                "POI_type": "tourist_attraction",
                "Buffer_range": 6000,
                "User_requirements": "has guided tours"
               }
    }
    
Inspecting the previous JSON object, the intent now is to derive the <Route> (i.e., the route from the origin to the destination).
Conducting intent recognition to the query:
    - Intent: computeRoutes (to find the path from origin to destination)
    - Entity: This would involves the origin, and the destination.
the expanded JSON object will be:

    {"intent": "searchNearby"
     "entity": {"Route_points": {"intent": "sampleRoutePoints",
                                 "entity": {"Route": {"intent": "computeRoutes",
                                                      "entity": <Origin>, <Destination>
                                                      }
                                            }, 
                                            "Num_route_points": 2
                                 }
                "POI_type": "tourist_attraction",
                "Buffer_range": 6000,
                "User_requirements": "has guided tours"
               }
    }
    
Inspecting the previous JSON object, the intent now is to derive the <Origin> and <Destination> of the route.
Conducting intent recognition to the query:
    - Intent: geocodeLocation (to get the geographical location of a place)
    - Entity: This would involves the origin (EG Ampol Coburg), and the destination (Monash Medical Centre).
the expanded JSON object will be:

    {"intent": "searchNearby"
         "entity": {"Route_points": {"intent": "sampleRoutePoints",
                                     "entity": {"Route": {"intent": "computeRoutes",
                                                          "entity": "Origin": {"intent": "geocodeLocation",
                                                                               "entity": "EG Ampol Coburg"}, 
                                                                    "Destination": {"intent": "geocodeLocation",
                                                                                    "entity": "Monash Medical Centre"}
                                                          }
                                                }, 
                                                "Num_route_points": 2
                                     }
                    "POI_type": "tourist_attraction",
                    "Buffer_range": 6000,
                    "User_requirements": "has guided tours"
                   }
    }
    

### Example 2

The geospatial query is: Find me a fitness center with a swimming pool along the way from Bunnings Box Hill to Eleven Tigers.

I need to find a fitness center from Bunnings Box Hill to Eleven Tigers.
I can intent to "searchAlongRoutes", since the query does not specify a buffer range.
Conducting intent recognition to the query:
    - Intent: searchAlongRoutes (sub-query to find places located along a specified route between two points)
    - Entity: This would include the POI type (i.e., fitness_center), the reference route (route), and the user semantic requirements (i.e. with a swimming pool)
the corresponding JSON object will be:

    {"intent": "searchAlongRoutes"
     "entity": <Route>, "POI_type": "fitness_center", "User_requirements": "with a swimming pool"
    }
    
    
Inspecting the previous JSON object, the intent now is to derive the <Route> (i.e., the route from the origin to the destination).
Conducting intent recognition to the query:
    - Intent: computeRoutes (to find the path from origin to destination)
    - Entity: This would involves the origin (Bunnings Box Hill), and the destination (Eleven Tigers).
the expanded JSON object will be:

    {"intent": "searchAlongRoutes"
     "entity": {"Route": {"intent": "computeRoutes",
                          "entity": <Origin>, <Destination>
                          }, 
     "POI_type": "fitness_center", 
     "User_requirements": "with a swimming pool"
    }
    
Inspecting the previous JSON object, the intent now is to derive the <Origin> and <Destination> of the route.
Conducting intent recognition to the query:
    - Intent: geocodeLocation (to get the geographical location of a place)
    - Entity: This would involves the origin (Bunnings Box Hill), and the destination (Eleven Tigers).
the expanded JSON object will be:

    {"intent": "searchAlongRoutes"
     "entity": {"Route": {"intent": "computeRoutes",
                          "entity": "Origin": {"intent": "geocodeLocation",
                                               "entity": "Bunnings Box Hill"}, 
                                    "Destination": {"intent": "geocodeLocation",
                                                    "entity": "Eleven Tigers"}
                          }
                "POI_type": "fitness_center", 
                "User_requirements": "with a swimming pool"
               }
    }
"""

In [ ]:
# ===================================
# Prompting Template (version 1.1)
# ===================================

system_prompt = """
You are a helpful assistant that helps breaking down complex geospatial queries into sub-queries in forms of JSON objects.
You will be given a complex geospatial query that requires both multi-hop geospatial reasoning and user semantic requirements analysis.
In order to structure the sub-queries systematically, you will be asked to perform a specific type of intent recognition called: Geospatial Intent Recognition.
You will be given a few examples showing how the Geospatial Intent Recognition works, try to follow and flexibly adapt to the structure of the example.

Please follow these three steps when breaking down the query:
    1. First, break down the query into multiple sub-queries. 
    2. Then, for each of the sub-query, generate appropriate reasoning thoughts to perform the Geospatial Intent Recognition.
    3. Finally, generate the corresponding JSON objects that help resolve the query.
    
Please comply the following rules when breaking down the query:

    1. Do not return the final JSON object until no sub-qustion can be further decomposed.
    2. Do not generate the final results until the reasoning for all sub-queries finished.
    3. For each sub-query, produce a reasoning section explaining how you interpret the intent and entities for that sub-query.
    4. For each sub-query, after reasoning is done, generate the intermediate JSON object.
    5. For each Geospatial Intent Recognition, try to first select one and only one "intent" strictly from this list: 
       ["computeRoutes", "searchNearby", "searchText", "searchAlongRoutes", "sampleRoutePoints", "geocodeLocation", "semanticComparison"]. If the intent cannot be described
       in any item within this list, you can construct your own intent.
    6. For each intent recognition, specify all the predicates that cannot be described by the "intent" object 
       into the "entity" object.
 
Please return a list of nested JSON objects as the final result.

--
## Examples:

### Example 1:

Input Query: Find me a tourist attraction that has guided tours along the way from EG Ampol Coburg to Monash Medical Centre, within 6km of the route.

To break down this query, we can identify the necessary sub-queries based on the locations and conditions specified in the user's request.
    1. Sub-query 1: Compute the route from EG Ampol Coburg to Monash Medical Centre.
    2. Sub-query 2: Find tourist attractions that has guided tours within a buffer of 6km along the route.

Now, based on the sub-queries, we conduct Geospatial Intent Recognition:

I need to compute the route from EG Ampol Coburg to Monash Medical Centre.
The intent here is to find a route that connects the specified origin and destination.
    - Intent: computeRoutes (to determine the path from the origin to destination)
    - Entity: This involves the origin (EG Ampol Coburg), the destination (Monash Medical Centre).
Resulting JSON object for sub-query 1 would be:

    {
      "intent": "computeRoutes",
      "entity": {
        "Origin": {
          "intent": "geocodeLocation",
          "entity": "EG Ampol Coburg"
        },
        "Destination": {
          "intent": "geocodeLocation",
          "entity": "Monash Medical Centre"
        }
      }
    }
    
Inspecting the previous JSON object, the task now is to derive <Route> (i.e. the route from the origin to the destination), <POI_TYPE> (i.e. places types) and <BUFFER_RANGE> (i.e. distance around the route) for the 'search_along_route' intent.
Conducting intent recognition to the query, the expanded JSON object will be:

    {"intent": "search_along_route"
     "entity": {"Route": {"intent": "compute_route",
                          "entity": <Origin>, <Destination>
                          }
                "POI_type": "tourist_attraction",
                "Buffer_range": 6000
               }
    }
    
Inspecting the previous JSON object, the task now is to derive the <Origin> and <Destination> for the 'compute_route' intent.
Conducting intent recognition to the query, the expanded JSON object will be:

    {"intent": "search_along_route"
     "entity": {"Route": {"intent": "compute_route",
                          "entity": "Origin": {"intent": "geo_code_location",
                                               "entity": "EG Ampol Coburg"}, 
                                    "Destination": {"intent": "geo_code_location",
                                                    "entity": "Monash Medical Centre"}
                          }
                "POI_type": "tourist_attraction",
                "Buffer_range": 6000
               }
    }
"""

user_prompt_instruction = """
You will be given a geospatial query, please execute Geospatial Intent Recognition to the query. Here is a few examples:
"""

user_prompt_input = f"Now, please execute Geospatial Intent Recognition to the following geospatial query: {query}"

In [2]:
# ===================================
# Prompting Template (version 1.0a) -- enhanced by openAI prompt editor
# ===================================

"""
You are a helpful assistant specializing in breaking down complex geospatial queries into hierarchical/nested sub-queries, using a JSON representation. For each input geospatial query that may involve multi-step (multi-hop) geospatial reasoning and user semantic requirements, follow these instructions:

1. **Decompose the Query**: 
    - Parse and break down the overall complex geospatial query into its core sub-queries.
    - Decompose recursively: For each sub-query, if further decomposition is possible, continue until the most atomic queries are reached.

2. **Geospatial Intent Recognition Reasoning**:
    - For each sub-query, before producing any JSON output, produce a reasoning section explaining how you interpret the intent and entities for that sub-query.
    - Use *one and only one* "intent" from the list: ["computeRoutes", "geocodeLocation", "searchNearby", "searchText", "searchAlongRoutes", "semanticComparison"] per sub-query. If none apply perfectly, you may construct a custom intent.
    - Assign any additional semantics, predicates, or requirements not captured by "intent" into the "entity" object.
    - **Do not present any conclusions or JSON output until you have reasoned through all sub-queries.** Always lead with your reasoning, followed by the conclusion.

3. **JSON Construction**:
    - Only after all reasoning is done for the current level, produce the corresponding nested JSON objects representing the structure of the sub-queries and their semantic requirements.
    - Each intent/entity object should be structured as:
        {
          "intent": <selected_intent_string>,
          "entity": { ... } // as detailed per your breakdown and reasoning
        }
    - Continue decomposing and nesting as demonstrated in the examples, ensuring no JSON is given until all reasoning is complete.

4. **Persistence and Chain-of-Thought**:
    - Continue decomposing recursively and do not produce any JSON or final answer unless all sub-queries have been fully analyzed and reasoned about.
    - Use a chain-of-thought reasoning process for each level and explain your logic step by step as you refine sub-queries.

5. **Formatting Requirements**:
    - **Output format:** For each breakdown, follow this order, repeated recursively as needed:
      - Reasoning step(s) for each sub-query (compose in narrative paragraphs or numbered points).
      - JSON object(s) for those sub-queries (do not output JSON until after corresponding reasoning).
    - **Length and detail:** 
      - Reasoning should clearly explain the logic at each decomposition stage.
      - JSON objects should use proper nesting, and use placeholders (e.g., <Origin>, <Destination>) where atomic values are not yet specified.
    - Final output should be a nested JSON object or a list of such objects, matching the structure in the provided examples.

6. **Edge Cases and Reminders**:
    - If any sub-query's intent does not fit the standard list, construct a suitable new intent and document your choice in reasoning.
    - Always ensure every predicate not captured by the standard intent is recorded in "entity".
    - Never shortcut reasoning; always explain each breakdown.
    - Never return a JSON object before all reasoning is complete for the respective branch of decomposition.

---

## Example

**Input query:**  
Find me a tourist attraction that has guided tours along the way from EG Ampol Coburg to Monash Medical Centre, within 6km of the route.

**Step 1: Reasoning**  
- The user wants a tourist attraction with guided tours, located along the driving route from EG Ampol Coburg to Monash Medical Centre, within a 6km buffer.  
- The highest-level intent is to "searchAlongRoutes" for POIs of type "tourist_attraction" (with the additional predicate "guided tours"), using a buffer of 6000 meters along the computed route.

**Step 2: Generate and Decompose Sub-queries**  
- "Route" needs to be defined: we must compute the route from the origin to the destination.
- "POI_type" is "tourist_attraction".
- "Buffer_range" is 6000.
- For the route, recursively, we need to geocode both origin and destination names.

**Step 3: Final JSON Output**   

{
  "intent": "searchAlongRoutes",
  "entity": {
    "Route": {
      "intent": "computeRoutes",
      "entity": {
        "Origin": {
          "intent": "geocodeLocation",
          "entity": "EG Ampol Coburg"
        },
        "Destination": {
          "intent": "geocodeLocation",
          "entity": "Monash Medical Centre"
        }
      }
    },
    "POI_type": "tourist_attraction",
    "Buffer_range": 6000,
    "Additional_predicate": "has guided tours"
  }
}

(**Real examples should include all levels of reasoning prior to presenting the JSON—in practice, supply a reasoning step before each piece of JSON is revealed.**)

---

**REMINDER:**  
- Always break down the query recursively, reason before producing JSON or any final answer, and use the specified list of intents or custom intents as your basis.  
- Output format: Narrative reasoning followed by a nested JSON object only after all relevant reasoning for that decomposition.  
- Never skip intermediate reasoning steps or present conclusions first; always reason before you conclude.

**Objective reminder:** Decompose complex geospatial queries into sub-queries; reason through intent recognition for each before outputting the corresponding nested JSON objects, following the above format strictly.vv
"""

### version 2: Reason-First-Generation-Later (RFGL)

In [1]:
# ===================================
# Prompting Template (version 2.0) 
# ===================================

"""
You are a helpful assistant specializing in breaking down complex geospatial queries into hierarchical/nested sub-queries, using a JSON representation. For each input geospatial query that may involve multi-step (multi-hop) geospatial reasoning and user semantic requirements, follow these three steps:

**Step 1: Query Decomposition**
    - Break down the overall complex geospatial query into its core sub-queries.
    - Keep decomposing each sub-query until the most atomic queries are reached.
    
**Step 2: Geospatial Intent Recognition**
    - For each of the sub-query, produce a reasoning section explaining how you interpret the intent and entities for that sub-query before generating the result JSON object.
    - Use one and only one "intent" from the list: ["computeRoutes", "geocodeLocation", "searchNearby", "searchText", "searchAlongRoutes", "sampleRoutePoints', semanticComparison"] per sub-query. You may construct a custom intent only if none of these apply.
    - If you choose "searchAlongRoutes" as an "intent", the entity should not contain a buffer range.
    - For each intent recognition, assign any additional semantics, predicates, or requirements not captured by "intent" into the "entity" object

**Step 3: Consturcting JSON objects**
    - Only if the reasoning for all sub-queries finished, generate the final JSON object.
    - Each intent/entity object should be structured as:
        {
          "intent": <selected_intent_string>,
          "entity": { ... }
        }
        
Please apply the following rules when breaking down the geospatial query:
    - Continue decomposing recursively and do not produce any JSON or final answer unless all sub-queries have been fully analyzed and reasoned about.
    - Use a chain-of-thought reasoning process for each level and explain your logic step by step as you refine sub-queries.

Please return the final output as a nested JSON object or a list of such objects.

--
## Example 1:

Please break down the following geospatial query: Find me a tourist attraction that has guided tours along the way from EG Ampol Coburg to Monash Medical Centre, within 6km of the route.

**Step 1: Query Decomposition**
The user wants a tourist attraction with guided tours, located along the driving route from EG Ampol Coburg to Monash Medical Centre, within a 6km buffer.
So I need to know the driving route from the origin (i.e. EG Ampol Coburg) to the destination (i.e. Monash Medical Centre).
Since I need to strictly search within a 6km buffer from this driving route, so I cannot search along route directly. Instead, I need to have some points along the route.
Once I have some points on the route, I need to know what are some tourist attractions within 6km buffer from the sampled points.

So the list of sub-queries for this geospatial query are:
["What is the driving route from EG Ampol Coburg to Monash Medical Centre?",
 "What are some points along this driving route?",
 "What are some tourist attractions within 6km buffer of the sampled points?"]

**Step 2: Geospatial Intent Recognition**
- The highest-level sub-query is: "What are some tourist attractions within 6km buffer of the sampled points?". The intent here is to "searchNearby" for POIs of type "tourist_attraction", using a buffer of 6000 meters along the computed route.
- The sampled points need to be defined. The next-level sub-query is "What are some points along this driving route?" The intent here is to 'SampleRoutePoints' along the driving route.
- The driving route need to be defined. The next-level sub-query is "What is the driving route from EG Ampol Coburg to Monash Medical Centre?". The intent here is to 'ComputeRoutes' from origin (EG Ampol Coburg) to destination (Monash Medical Centre).
- This involves two separate components for geocoding the origin and destination, in order to get their geographical locations.

**Step 3: Consturcting JSON objects** 

{
  "intent": "searchNearby",
  "entity": {
    "RoutePoints": {
          "intent": "SampleRoutePoints",
          "entity": {"Route": {
              "intent": "computeRoutes",
              "entity": {
                "Origin": {
                  "intent": "geocodeLocation",
                  "entity": "EG Ampol Coburg"
                },
                "Destination": {
                  "intent": "geocodeLocation",
                  "entity": "Monash Medical Centre"
                }
              }
            }
        }
    },
    "Buffer_range": 6000,
    "POI_type": "tourist_attraction",
    "Additional_predicate": "has guided tours"
  }
}


"""


'\nYou are a helpful assistant specializing in breaking down complex geospatial queries into hierarchical/nested sub-queries, using a JSON representation. For each input geospatial query that may involve multi-step (multi-hop) geospatial reasoning and user semantic requirements, follow these three steps:\n\n**Step 1: Query Decomposition**\n    - Break down the overall complex geospatial query into its core sub-queries.\n    - Keep decomposing each sub-query until the most atomic queries are reached.\n\n**Step 2: Geospatial Intent Recognition**\n    - For each of the sub-query, produce a reasoning section explaining how you interpret the intent and entities for that sub-query before generating the result JSON object.\n    - Use one and only one "intent" from the list: ["computeRoutes", "geocodeLocation", "searchNearby", "searchText", "searchAlongRoutes", "sampleRoutePoints\', semanticComparison"] per sub-query. You may construct a custom intent only if none of these apply.\n    - If you

In [ ]:
# ===================================
# Prompting Template (version 2.1)
# ===================================
"""
You are a helpful assistant specializing in breaking down complex geospatial queries into hierarchical/nested sub-queries, using a JSON representation. For each input geospatial query that may involve multi-step (multi-hop) geospatial reasoning and user semantic requirements, follow these three steps:

**Step 1: Query Decomposition**
    - Break down the overall complex geospatial query into its core sub-queries.
    - Keep decomposing each sub-query until the most atomic queries are reached.
    
**Step 2: Geospatial Intent Recognition**
    - For each of the sub-query, produce a reasoning section explaining how you interpret the intent and entities for that sub-query before generating the result JSON object.
    - Use one and only one "intent" from the list: ["computeRoutes", "geocodeLocation", "searchNearby", "searchText", "searchAlongRoutes", "sampleRoutePoints', semanticComparison"] per sub-query. You may construct a custom intent only if none of these apply.
    - If you choose "searchAlongRoutes" as an "intent", the entity should not contain a buffer range.
    - For each intent recognition, assign any additional semantics, predicates, or requirements not captured by "intent" into the "entity" object

**Step 3: Consturcting JSON objects**
    - Only if the reasoning for all sub-queries finished, generate the final JSON object.
    - Each intent/entity object should be structured as:
        {
          "intent": <selected_intent_string>,
          "entity": { ... }
        }
        
Please apply the following rules when breaking down the geospatial query:
    - Continue decomposing recursively and do not produce any JSON or final answer unless all sub-queries have been fully analyzed and reasoned about.
    - Use a chain-of-thought reasoning process for each level and explain your logic step by step as you refine sub-queries.

Please return the final output as a nested JSON object or a list of such objects.

--
## Example

**Input Query:**  
Find me a tourist attraction that has guided tours along the way from EG Ampol Coburg to Monash Medical Centre, within 6km of the route.

**Step 1: Query Decomposition**  
The user wants to identify a tourist attraction with guided tours near a driving route from EG Ampol Coburg to Monash Medical Centre, specifically within a 6km buffer zone. This requires determining the route, sampling points along the route, then searching for attractions near those points, and ensuring they offer guided tours.

Sub-queries identified:
1. "What is the driving route from EG Ampol Coburg to Monash Medical Centre?"
2. "What are some points along this driving route?"
3. "What are some tourist attractions within 6km of those points with guided tours?"

**Step 2: Geospatial Intent Recognition**  
- For "What is the driving route from EG Ampol Coburg to Monash Medical Centre?":
    - Intent: "computeRoutes". The geocoded locations of the origin and destination are required, so two further sub-queries to geocode those names.
- For "What are some points along this driving route?":
    - Intent: "sampleRoutePoints". This uses the computed route from above.
- For "What are tourist attractions within 6km of the points with guided tours?":
    - Intent: "searchNearby". Entity includes POI type "tourist_attraction", buffer range 6000m, and predicate "has guided tours", based on the sampled points.

**Step 3: Consturcting JSON objects** 

{
  "intent": "searchNearby",
  "entity": {
    "RoutePoints": {
          "intent": "SampleRoutePoints",
          "entity": {"Route": {
              "intent": "computeRoutes",
              "entity": {
                "Origin": {
                  "intent": "geocodeLocation",
                  "entity": "EG Ampol Coburg"
                },
                "Destination": {
                  "intent": "geocodeLocation",
                  "entity": "Monash Medical Centre"
                }
              }
            }
        }
    },
    "Buffer_range": 6000,
    "POI_type": "tourist_attraction",
    "Additional_predicate": "has guided tours"
  }
}

## 1.2 Prompting Style 2 -- modular (process-guaranteed)

# 2. Sub-query-to-API

In [3]:
import os
from dotenv import load_dotenv
import requests
import re
import time
from openai import OpenAI
from collections import defaultdict
import json

from itertools import islice

import polyline
from shapely.geometry import LineString
from pyproj import Transformer
import matplotlib.pyplot as plt
import numpy as np

# ----------------
# Set up API Keys
# ----------------

load_dotenv()  # load .env file

openai_key = os.getenv("OPENAI_API_KEY")
google_maps_key = os.getenv("GOOGLE_MAPS_API_KEY")

## 2.1 Extract Sub-queries from LLM Response

In [4]:
def extract_stage2_json(llm_response: str):
    """
    Extract the JSON object of Stage 2 (sub_queries) from the LLM response.
    Works even with nested braces and commas.
    """
    start_keyword = '"sub_queries"'
    start_idx = llm_response.find(start_keyword)
    if start_idx == -1:
        raise ValueError("Cannot find 'sub_queries' in the response")
    
    # Find the opening brace before "sub_queries"
    brace_start = llm_response.rfind("{", 0, start_idx)
    if brace_start == -1:
        raise ValueError("Cannot find opening brace for Stage 2 JSON")
    
    # Iterate to find the matching closing brace
    brace_count = 0
    for i in range(brace_start, len(llm_response)):
        if llm_response[i] == "{":
            brace_count += 1
        elif llm_response[i] == "}":
            brace_count -= 1
            if brace_count == 0:
                brace_end = i + 1
                break
    else:
        raise ValueError("No matching closing brace found")
    
    json_str = llm_response[brace_start:brace_end]
    json_obj = json.loads(json_str)
    
    # Convert to Python dict
    return json_obj["sub_queries"]

## 2.2 Convert Sub-queries into APIs

In [19]:
# Example flattened sub-queries array
sub_queries = [
    {
        "id": "q1",
        "intent": "geocodeLocation",
        "entity": {"location": "Greater Melbourne"}
    },
    {
        "id": "q2",
        "intent": "searchNearby",
        "entity": {
            "POI_type": "train station",
            "Reference_location": "$q1",
            "radius": 5000
        }
    }
]

sub_queries_2 = [
    {
      "id": "q1",
      "intent": "geocodeLocation",
      "entity": {
        "place": "Bunnings Box Hill"
      }
    },
    {
      "id": "q2",
      "intent": "geocodeLocation",
      "entity": {
        "place": "Eleven Tigers"
      }
    },
    {
      "id": "q3",
      "intent": "computeRoutes",
      "entity": {
        "origin": "$q1",
        "destination": "$q2"
      }
    },
    {
      "id": "q4",
      "intent": "searchAlongRoutes",
      "entity": {
        "route": "$q3",
        "POI_type": "fitness_center", 
        "user_requirements": "with a swimming pool"
      }
    }
  ]

sub_queries_3 = [
    {
      "id": "q1",
      "intent": "geocodeLocation",
      "entity": {
        "place": "EG Ampol Coburg"
      }
    },
    {
      "id": "q2",
      "intent": "geocodeLocation",
      "entity": {
        "place": "Monash Medical Centre"
      }
    },
    {
      "id": "q3",
      "intent": "computeRoutes",
      "entity": {
        "origin": "$q1",
        "destination": "$q2"
      }
    },
    {
      "id": "q4",
      "intent": "sampleRoutePoints",
      "entity": {
        "route": "$q3",
        "num_route_points": 2
      }
    },
    {
      "id": "q5",
      "intent": "searchNearby",
      "entity": {
        "reference_location": "$q4",
        "POI_type": "tourist_attraction",
        "buffer_range": 6000,
        "user_requirements": "has guided tours"
      }
    }
  ]

In [5]:
# Helper to resolve $id placeholders
def resolve_placeholders(entity, results):
    if isinstance(entity, str) and entity.startswith("$"):
        ref_id = entity[1:]
        return results[ref_id]
    elif isinstance(entity, dict):
        return {k: resolve_placeholders(v, results) for k, v in entity.items()}
    elif isinstance(entity, list):
        return [resolve_placeholders(v, results) for v in entity]
    return entity

In [6]:
# ------------------------------------------
#  Helper functions for API call functions
# ------------------------------------------

### geocode_location helpers ###
#------------------------------#

def split_name_address(s):
    """
    Extract <name> and <address> from strings like
    'Name at Address'
    """
    pattern = r"^(.*?)\s+at\s+(.*)$"

    m = re.match(pattern, s, flags=re.IGNORECASE)
    if m:
        name, address = m.groups()
        return name.strip(), address.strip()
    else:
        # no match, return the whole string as name, None as address
        return s.strip(), None

# Helper to convert displayed name of locations into places ids
def resolve_location(name, location_bias=(-37.8136, 144.9631), radius=50000):
    """
    Resolve a human-readable location name (within Melbourne) into
    place_id and lat/lng using Google Places API.
    
    location_bias: (lat, lng) used as the search center (Melbourne CBD default)
    radius: in meters
    """
    lat, lng = location_bias
    location_name, address = split_name_address(name)
    
    url = "https://places.googleapis.com/v1/places:searchText"

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": google_maps_key, 
        "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.location,places.reviews" # space is not allowed!!!
    }

    body = {
        "textQuery": location_name,
        "locationBias": {
          "circle": {
            "center": {
              "latitude": lat,
              "longitude": lng
            },
            "radius": 50000
          }
        }
    }
    
    ### Post Request ###
    response = requests.post(url, headers=headers, json=body)
    response.raise_for_status()
    data = response.json()

    ### Parse Response ###
    result = None
    if not data["places"]:
        return result
    
    # if no address is specified, we return the first place in the response
    if address is None:
        result = data["places"][0] 
    else:  # otherwise we check the address
        for place in data["places"]:
            if place["formattedAddress"] == address:  # match on address
                result = place
        if result is None:
            return None
        
    time.sleep(0.5) 
    
    return result


### compute_routes helpers ###
#------------------------------#

def parse_route(route, origin, dest):
    
    '''
    This function parses the route object responsed by Google Maps API and return a list of encoded polylines 
        Input:
            route: a json object
            origin: origin of the entire route
            dest: destination of the entire route
        Output:
            polylines: a list of 1 or 2 encoded polylines
    '''
    
    polylines = []
    
    # if only one leg within a route
    if 'legs' not in route:
        polylines.append({'encodedPolyline': route['polyline']['encodedPolyline'],
                          'origin': origin,
                          'destination': dest,
                          'distanceMeters': route['distanceMeters'],
                          'duration': route['duration']})
    
    # if two legs within a route
    else:
        legs = route['legs']
        for leg in legs:
            polylines.append({'encodedPolyline': leg['polyline']['encodedPolyline'],
                          'origin': leg['startLocation']['latLng'],
                          'destination': leg['endLocation']['latLng'],
                          'distanceMeters': leg['distanceMeters'],
                          'duration': leg['duration']})
            
    return polylines

In [7]:
# API call functions for each intent

def geocode_location(location):
    
    '''
    Input: location name (NL text)
    Output: single loc dict -- e.g., {'latitude': -37.7656718, 'longitude': 144.98295629999998} 
    '''
    
    loc_dict = resolve_location(location)
    
    if loc_dict is not None:
        return loc_dict['location']
    return None

    ##### how to handle none cases ??? #####

def search_nearby(ref_loc, POI_type, radius=1000):
    
    results = []
    
    POI_type_list = []
    POI_type_list.append(POI_type)
    
    url = "https://places.googleapis.com/v1/places:searchNearby"

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": google_maps_key, 
        "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.location,places.reviews" # space is not allowed!!!
    }

    body = {
        "includedTypes": POI_type_list,
        "maxResultCount": 20,
        "locationRestriction": {
            "circle": {
            "center": {
                "latitude": ref_loc["latitude"],
                "longitude": ref_loc["longitude"]},
            "radius": radius
            }
        }
    }
    
    ### Post Request ###
    response = requests.post(url, headers=headers, json=body)
    response.raise_for_status()
    
    ### Parse Response ###
    if response.status_code == 200:
        data = response.json() # parse into python dict
    
        for place in data.get('places', []):
            results.append(place)
     
    time.sleep(0.5)
    
    return results
    

def compute_routes(origin, dest, waypoint=None):  # travel mode = DRIVE
    
    '''
    Inputs: 
        origin: {lat, lng} dict
        dest: {lat, lng} dict
        waypoint (optional): {lat, lng} dict

    Output: 
        list(): encoded_polylines -- a list (1 or 2) encoded polylines
    '''
    
    result = []
    
    ### Forming Request ###
    url = "https://routes.googleapis.com/directions/v2:computeRoutes"
    
    # if the question does not contain waypoint
    if waypoint is None:
        headers = {
            "Content-Type": "application/json",
            "X-Goog-Api-Key": google_maps_key, 
            "X-Goog-FieldMask": "routes.duration,routes.polyline.encodedPolyline,routes.distanceMeters"
        } 

        body = {
                "origin":{
                "location":{
                  "latLng":{
                    "latitude": origin['latitude'],
                    "longitude": origin['longitude']
                  }
                }
              },
              "destination":{
                "location":{
                  "latLng":{
                    "latitude": dest['latitude'],
                    "longitude": dest['longitude']
                  }
                }
              },
              "travelMode": "DRIVE",  # we focus on driving queries, if distance too small, remove from the list
              "languageCode": "en-US",
              "units": "METRIC"
        }
    
    # if the question contains a waypoint
    if waypoint is not None:
        headers = {
            "Content-Type": "application/json",
            "X-Goog-Api-Key": google_maps_key, 
            "X-Goog-FieldMask": "routes.duration,routes.distanceMeters,routes.legs" # specify the route legs to return all 2 legs
        } 

        body = {
                "origin":{
                "location":{
                  "latLng":{
                    "latitude": origin['latitude'],
                    "longitude": origin['longitude']
                  }
                }
              },
              "destination":{
                "location":{
                  "latLng":{
                    "latitude": dest['latitude'],
                    "longitude": dest['longitude']
                  }
                }
              },
              "intermediates": [
              {
                "location":{
                  "latLng":{
                    "latitude": waypoint['latitude'],
                    "longitude": waypoint['longitude']
                  }
                }
              }
            ],
              "travelMode": "DRIVE",  # we focus on driving queries, if distance too small, remove from ths lis
              "languageCode": "en-US",
              "units": "METRIC"
        }
    
    
    ### Post Request ###
    response = requests.post(url, headers=headers, json=body)
    
    ### Check and Parse Response ###
    if response.status_code == 200:
        data = response.json() # parse into python dict
        
        route = data.get('routes', [])[0]  #since there is no alternative routes, so we take the only route

    # parse Google Maps response
    poly_list = parse_route(route, origin, dest)
    
    time.sleep(0.5)
    
    return poly_list

def search_along_routes(route, textQuery):
    
    results = []
    
    for r in route:
        polyline = r["encodedPolyline"]
    
        ### Forming Request ###
        url = "https://places.googleapis.com/v1/places:searchText"

        headers = {
            "Content-Type": "application/json",
            "X-Goog-Api-Key": google_maps_key, 
            "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.location,places.reviews" # space is not allowed!!!
        }

        body = {
            "textQuery" : textQuery,
            "searchAlongRouteParameters": {
              "polyline": {
                "encodedPolyline": polyline
              }
            }
        }
        
        ### Post Request ###
        response = requests.post(url, headers=headers, json=body)
        response.raise_for_status()

        ### Parse Response ###
        if response.status_code == 200:
            data = response.json() # parse into python dict

            for place in data.get('places', []):
                results.append(place)
     
    time.sleep(0.5)
    
    return results

def sample_route_points(route, n_points=2):
    """
    Sample a fixed number of points evenly along each encoded polyline.

    inputs:
        route : list[str]
            A list of encoded polylines (Google format).
        n_points : int
            Number of points to sample along each polyline.

    outputs:
        list[tuple[float, float]]
            A flat list of sampled points as (lat, lon) tuples.
    """
    sampled_points = []

    # Initialize transformer (coordinate conversion)
    transformer = Transformer.from_crs("epsg:7844", "epsg:3111", always_xy=True)  # degree --> meters

    for r in route:
        
        encoded_polyline = r['encodedPolyline']
        # Decode polyline to list of (lat, lon) coordinates
        coords = polyline.decode(encoded_polyline)
        
        # Project coordinates from 7844 to 3111
        line_coords_proj = [transformer.transform(lon, lat) for lat, lon in coords]
        line_proj = LineString(line_coords_proj)

        # Compute distances along line where points will be sampled
        distances = np.linspace(0, line_proj.length, n_points + 2)[1:-1]  # we exclude origin and dest here

        # Interpolate points along the projected line
        sampled_points_proj = [line_proj.interpolate(d) for d in distances]

        # Transform points back to lat/lon
        sampled_points += [transformer.transform(p.x, p.y, direction="INVERSE") for p in sampled_points_proj]
    
    # Turn into list of ref_locations (lat, lng dict)
    final_point_list = []
    for pt in sampled_points:
        pt = {"latitude": pt[1], "longitude": pt[0]}
        final_point_list.append(pt)
        
    return final_point_list

In [8]:
# -------------------------
# Helper Functions for entity parsing
# -------------------------

### General helpers ###
SUPPORTED_PLACE_TYPES = {
    "gas_station","parking", "car_dealer", "car_repair","car_wash",
    "library", "school","university",
    "amusement_center","park", "tourist_attraction",
    "atm", "bank",
    "bakery","bar","cafe", "restaurant","police","post_office",
    "hospital","pharmacy","spa",
    "apartment_building", "housing_complex",
    "public_bathroom",
    "hotel",
    "beach",
    "church",
    "barber_shop","beauty_salon","florist",
    "insurance_agency","laundry","lawyer",
    "real_estate_agency","travel_agency","veterinary_care",
    "book_store","clothing_store","convenience_store","grocery_store", "store","supermarket",
    "fitness_center","gym","stadium",
    "airport","bus_station","train_station", "transit_station",
    "establishment"  # fallback default
}

def normalize_and_validate_POI_type(user_input: str, fallback: str = "establishment"):
    """
    Convert a human-readable place type to Google Maps API type format,
    validate against supported types, and fallback if unsupported.
    """
    normalized = user_input.strip().lower().replace(" ", "_")
    return normalized if normalized in SUPPORTED_PLACE_TYPES else fallback

### Geocode Location Parsing Helper ###
def extract_location(entity):
    """Return the only value inside an entity dict, ignoring the key."""
    if isinstance(entity, dict):
        return next(iter(entity.values()))
        # if there are multiple keys, also return only the first one
    return entity  # if it's already a string

### Nearby Search Parsing Helper ###
def extract_reference_location(entity, fallback=None):
    for key, val in entity.items():
        if re.search(r'ref.*loc', key, re.I):  # any 'ref'...'loc' case-insensitive
            return val
    return fallback


### Semantic Comparison ###
#------------------------------#
def concatenate_reviews(reviews):
    
    total_rating = 0
    rating_count = 0
    concatenated_reviews = ""
    
    if not reviews:  # the candidate does not have any review 
        return concatenated_reviews
        
    for r in reviews:
        
        total_rating += int(r.get('rating', 0))
        rating_count += 1

        # handle reviews without 'text'
        review_text = ""
        if 'text' in r:
            # if r['text'] itself is a dict with 'text' inside:
            if isinstance(r['text'], dict):
                review_text = r['text'].get('text', "")
            else:  # if r['text'] is already a string
                review_text = r['text']
        concatenated_reviews += review_text + " "

    # avoid division by zero if no reviews
    average_rating = round(total_rating / rating_count, 1) if rating_count else 0

    concatenated_reviews = f"Rating [{average_rating}/5] " + concatenated_reviews.strip()

    return concatenated_reviews

'''
Truncate the reviews into only around 512 tokens.
'''
def truncate_review(review_text, max_tokens=512):
    words = review_text.split()  # rough estimate of tokens
    truncated = " ".join(words[:int(max_tokens*0.75)])  # ~512 tokens
    return truncated

def get_embedding(client, text, model="text-embedding-3-small"):
    
    # print(text)
    response = client.embeddings.create(
        input=text,
        model=model
    )
    return np.array(response.data[0].embedding)  # output embedding dimension: (, 1536)

def get_embedding_batch(client, text_list, model="text-embedding-3-small", batch_size=50):
    """
    Embed multiple texts in batches to reduce API calls.
    
    Args:
        text_list: list of strings
        model: embedding model
        batch_size: number of texts per API call
    Returns:
        np.array of embeddings
    """
    all_embeddings = []
    
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        response = client.embeddings.create(
            input=batch,
            model=model
        )
        # each response.data[i].embedding corresponds to batch[i]
        batch_embeddings = [np.array(item.embedding) for item in response.data]
        all_embeddings.extend(batch_embeddings)
    
    return np.stack(all_embeddings)

# simple chunk function by words
def chunk_text(text, chunk_size=150, overlap=20):
    """
    Split text into overlapping word chunks.
    chunk_size: number of words per chunk
    overlap: number of words overlapping between chunks
    """
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap  # move with overlap
    return chunks

# Vectorized cosine similarity
def cosine_sim_matrix(query_emb, candidate_emb_matrix):
    query_norm = np.linalg.norm(query_emb)
    candidate_norms = np.linalg.norm(candidate_emb_matrix, axis=1)
    sims = candidate_emb_matrix @ query_emb
    sims /= (candidate_norms * query_norm + 1e-8)
    return sims

def chunk_embedding_score(query_emb, review):

    """
    Compute similarity between query and review using batched embeddings.
    """
    if not review.strip():
        return 0.0

    truncated_review = truncate_review(review)
    chunks = chunk_text(truncated_review, chunk_size=150, overlap=20)
    if not chunks:
        return 0.0

    # embed chunks in batches
    chunk_embeddings = get_embedding_batch(chunks, batch_size=50)

    # vectorized similarity
    sims = cosine_sim_matrix(query_emb, chunk_embeddings)

    return sims.max()

'''
This function get a question and derive candidates locations for the ground-truth answer

    Inputs:
    ---------
        client: OpenAI client
        qa_key: the geospatial queries
        qa_val: all attributes of the query (the potential dataset columns)
    
    Output:
    ---------
        top_5_candidates: a list of top-5 related locations' names
'''

def semantic_matching(client, user_req, candidates):
    
    ## Query Embedding (Only semantic part)
    user_req_embed = get_embedding(client, user_req)
    
    ## Candidate (reviews) Embedding [Chunk-embedding] + Cosine Similarity Comparison
    all_chunks = []
    chunk_owner = []  # keep track of which candidate each chunk belongs to

    for cand in candidates:
        reviews = cand.get('reviews', "")
        concatenated_reviews = concatenate_reviews(reviews)
        truncated_review = truncate_review(concatenated_reviews)
        chunks = chunk_text(truncated_review, chunk_size=150, overlap=20)
        for ch in chunks:
            all_chunks.append(ch)
            chunk_owner.append(cand["id"])  # each location has unique id,  can use for key
            
    if not all_chunks:
        return []

    # One big embedding call:
    chunk_embs = get_embedding_batch(client, all_chunks, batch_size=50)

    # Group embeddings by candidate:
    cand_to_embs = defaultdict(list)
    for emb, owner in zip(chunk_embs, chunk_owner):
        cand_to_embs[owner].append(emb)
    
    # Now compute similarities:
    cand_score = []
    for cand_key, emb_list in cand_to_embs.items():
        sims = cosine_sim_matrix(user_req_embed, np.vstack(emb_list))
        score = sims.max()
        cand_score.append((cand_key, score))

    cand_score.sort(key=lambda x: x[1], reverse=True)  
    
    # Filter top-5 relevant candidates as results
    top_5 = cand_score[:5]
    top_5_dict = dict(top_5)
    
    return [cand for cand in candidates if cand['id'] in top_5_dict.keys()]

In [157]:
# =================================
# Execute sub-queries sequentially
# =================================

OpenAI_client = OpenAI(api_key=openai_key)
results = {}

# sub_queries

for sq in sub_queries_3:
    intent = sq["intent"]
    entity = resolve_placeholders(sq["entity"], results)

    if intent == "geocodeLocation":
        location = extract_location(entity)
        results[sq["id"]] = geocode_location(location)

    elif intent == "searchNearby":
        poi = normalize_and_validate_POI_type(entity["POI_type"])
        ref_loc = extract_reference_location(entity)
        radius = entity.get("radius", 10000)
        
        ## Handle multiple reference locations ##
        if isinstance(ref_loc, list):
            places_nearby = []
            for loc in ref_loc:
                places_nearby.extend(search_nearby(loc, poi, radius))
        else:
            places_nearby = search_nearby(ref_loc, poi, radius)
        
        ## Handle User Semantic Requirements ##
        if "user_requirements" in entity:
            user_req = entity["user_requirements"]
            matched_places = semantic_matching(OpenAI_client, user_req, places_nearby)
            results[sq["id"]] = matched_places
        else:
            results[sq["id"]] = places_nearby
        
    elif intent == "computeRoutes":
        origin = entity["origin"]
        destination = entity["destination"]
        if "waypoint" in entity:
            waypoint = entity["waypoint"]
            route = compute_routes(origin, destination, waypoint=waypoint)
        else:
            route = compute_routes(origin, destination)
            
        results[sq["id"]] = route
        
    elif intent == "searchAlongRoutes":
        route = entity["route"]  # a list of route(s)
        textQuery = entity["POI_type"]
        
        places_along_routes = []
        for r in route:
            places = search_along_routes(route, textQuery)
            places_along_routes.extend(places)
        
        ## Handle User Semantic Requirements ##
        if "user_requirements" in entity:
            user_req = entity["user_requirements"]
            matched_places = semantic_matching(OpenAI_client, user_req, places_along_routes)
            results[sq["id"]] = matched_places
        else:
            results[sq["id"]] = places_along_routes
    
    elif intent == "sampleRoutePoints":
        route = entity["route"]
        num_route_points = entity["num_route_points"]
        sampled_points = sample_route_points(route, n_points=num_route_points)
        results[sq["id"]] = sampled_points

# results now holds the output for each sub-query by id
print(results)

{'latitude': -37.828260000098304, 'longitude': 144.95804886339258} tourist_attraction 10000
{'latitude': -37.86526619187425, 'longitude': 145.0610057056886} tourist_attraction 10000
{'q1': {'latitude': -37.753299999999996, 'longitude': 144.945}, 'q2': {'latitude': -37.9199663, 'longitude': 145.12423809999999}, 'q3': [{'encodedPolyline': 'juleFiqtsZ@VINSHG?qC`f@UjDOjDIfANVZRdAa@rB_AbEaBjBa@VNlFsAfAMh@OdCYlF[dCCxHPjDLhN\\`^nAbQx@`EFdI?jBI`DYpAQhEcA~DoAbHsCrDyAlDu@~AQzLa@lBCxA@jBNvCh@ZNh@ZtB|@fDnBtFlDhBz@~Bv@bB^zH`AtLpAxCFhBEhCYxHoAlAGn@GzBCdCLhBRdDr@~Bx@pB~@rBvAnElD`Al@xBbA~Bj@hCVfC@zAK~RoBdEO|CCfHd@tDf@hFnAfXlHnIzBrD~@rCb@`ALtAIr@MhAk@f@e@V]b@}@XiAFiA?cB{@sNc@mMSoKAoB@sCL}Dh@{K@YE_@DuAl@sIvAeNVqB`CiPf@}D~ByP\\kEHyB?yAE{CIuA[sCmDkQkAeGi@kDm@wEs@}DsCaNq@uC{@oE]gDM_BQkCKwAWuGKwKFqFJaSb@qU~@iZ`@gG`@aDj@kDRkAt@aCz@cCzAuDnF}Kp@{AfBuEl@gB~@iEjAqGbAqHf@wC~@_G@E|A_J^qEDyBAoAOwBUsAg@cC_B{FO{@U_CEqB?kAL}B^kDjBgNh@yF^cGReBr@oC\\}@t@uAvAiBzJcIvJmIz@w@fBoBn@aAj@oAj@qBZiBH}@D{BGkBIkAgAkIMkAK{B@gCHyBZg

In [160]:
# =================
# Parse Results
# =================

final_subQuery_id = sub_queries_3[-1]["id"]  # last sub-query
final_subQuery_results = results[final_subQuery_id]

answers = []
for place in final_subQuery_results:
    answers.append({"displayName": place["displayName"]['text'],
                    "location": place["location"],
                    "id": place["id"],
                    "formattedAddress": place["formattedAddress"]
                    })

print(answers)

[{'displayName': 'SEA LIFE Melbourne Aquarium', 'location': {'latitude': -37.820627, 'longitude': 144.95821999999998}, 'id': 'ChIJ_QzhcE1d1moR9rtoKagNPOI', 'formattedAddress': 'King St, Melbourne VIC 3000, Australia'}, {'displayName': 'Royal Exhibition Building', 'location': {'latitude': -37.8046891, 'longitude': 144.9716502}, 'id': 'ChIJcd7_l9lC1moRN3U9OQmxoEQ', 'formattedAddress': '9 Nicholson St, Carlton VIC 3053, Australia'}, {'displayName': 'Melbourne Town Hall', 'location': {'latitude': -37.815085599999996, 'longitude': 144.96656529999998}, 'id': 'ChIJeRyD3rVC1moRY20R7vTeJxk', 'formattedAddress': '90/130 Swanston St, Melbourne VIC 3000, Australia'}, {'displayName': 'ACMI', 'location': {'latitude': -37.8175974, 'longitude': 144.96868809999998}, 'id': 'ChIJT8T86bZC1moRpBiRUC3lABM', 'formattedAddress': 'Federation Square, Flinders St, Melbourne VIC 3000, Australia'}, {'displayName': 'Parliament of Victoria', 'location': {'latitude': -37.811048199999995, 'longitude': 144.973813899999

**Some intermediate result format**

In [22]:
#res = search_nearby(-37.7656718, 144.98295629999998, 'fitness_center', radius=5000)

In [11]:
res

[{'id': 'ChIJJdSq57Rc1moRdDb4tcLyxjc',
  'formattedAddress': '14 Dawson St, Brunswick VIC 3056, Australia',
  'location': {'latitude': -37.7711, 'longitude': 144.9597},
  'displayName': {'text': 'Brunswick Baths', 'languageCode': 'en'}},
 {'id': 'ChIJVQy9RSJD1moRoNrBOS1zCLw',
  'formattedAddress': '160 Alexandra Parade, Fitzroy VIC 3065, Australia',
  'location': {'latitude': -37.7938293, 'longitude': 144.9810556},
  'displayName': {'text': 'Fitzroy Swimming Pool', 'languageCode': 'en'}},
 {'id': 'ChIJu_CTZ_VD1moRpkyCtkNi-so',
  'formattedAddress': 'Floor 1/62-70 Johnston St, Fitzroy VIC 3065, Australia',
  'location': {'latitude': -37.7983279, 'longitude': 144.97679130000003},
  'displayName': {'text': 'Upstate Studios', 'languageCode': 'en'}},
 {'id': 'ChIJC5uXlyhD1moRgcN1IfEeHo4',
  'formattedAddress': '216/248 Rathdowne St, Carlton VIC 3053, Australia',
  'location': {'latitude': -37.793322499999995, 'longitude': 144.9714371},
  'displayName': {'text': 'Carlton Baths', 'languageCod

In [19]:
#route_res = compute_routes({"latitude": -37.7656718, "longitude": 144.98295629999998}, {"latitude": -37.872701, "longitude": 145.0233725}, waypoint=None)

In [20]:
route_res

[{'encodedPolyline': '~aoeFs|{sZHJ@F?\\PAjEf@bBNT?MyEHyAfAiN\\cGrB_YtFx@bZjD|@HlC^fEf@XAbBJlA[x@WxEeAb@Sj@q@dFmHl@m@lAiAJ]hAeUhO|A`ANz@\\hBfBVZjAjAVATCPM`@g@r@a@f@If@?~A`@jCd@vCZr@DTA`CLp@@nBLd@PvU`C^HHEtB`@pFj@rJr@bHn@fD`@lUxB~SnBdNrAvDZtMtAdKdAdBTfe@nEnIp@x@DlPbBlFf@~El@bCZjE^hA?hJeJfBqB~@_Bl@yAb@eB~@_GbB}L`AoH@aBIq@i@oDSoCYsBw@gDOyA@yAJ}@Tu@^q@rA{BTy@Fs@EWEmAOiC@w@L{@L[h@s@\\Mt@KX@LFzBTbBVpVhCzEp@T?lZlDhMzAxY|B~@DrUhCnLlAlAoP~@yN^kFd@uGVmDnB{Z\\aCZmA|Xqz@xAkEx@VpBXVJpS|B`QhB[lFf@F',
  'origin': {'latitude': -37.7656718, 'longitude': 144.98295629999998},
  'destination': {'latitude': -37.872701, 'longitude': 145.0233725},
  'distanceMeters': 16179,
  'duration': '2097s'}]

In [25]:
# search_along_route_res = search_along_routes(
#     [{'encodedPolyline': '~aoeFs|{sZHJ@F?\\PAjEf@bBNT?MyEHyAfAiN\\cGrB_YtFx@bZjD|@HlC^fEf@XAbBJlA[x@WxEeAb@Sj@q@dFmHl@m@lAiAJ]hAeUhO|A`ANz@\\hBfBVZjAjAVATCPM`@g@r@a@f@If@?~A`@jCd@vCZr@DTA`CLp@@nBLd@PvU`C^HHEtB`@pFj@rJr@bHn@fD`@lUxB~SnBdNrAvDZtMtAdKdAdBTfe@nEnIp@x@DlPbBlFf@~El@bCZjE^hA?hJeJfBqB~@_Bl@yAb@eB~@_GbB}L`AoH@aBIq@i@oDSoCYsBw@gDOyA@yAJ}@Tu@^q@rA{BTy@Fs@EWEmAOiC@w@L{@L[h@s@\\Mt@KX@LFzBTbBVpVhCzEp@T?lZlDhMzAxY|B~@DrUhCnLlAlAoP~@yN^kFd@uGVmDnB{Z\\aCZmA|Xqz@xAkEx@VpBXVJpS|B`QhB[lFf@F',
#   'origin': {'latitude': -37.7656718, 'longitude': 144.98295629999998},
#   'destination': {'latitude': -37.872701, 'longitude': 145.0233725},
#   'distanceMeters': 16179,
#   'duration': '2097s'}], "ATM"
# )

In [28]:
search_along_route_res

[{'id': 'ChIJScpuOXxD1moRzGy1Mg-dGjw',
  'formattedAddress': '290 State Route 29, Northcote VIC 3070, Australia',
  'location': {'latitude': -37.771682999999996, 'longitude': 144.9988339},
  'displayName': {'text': 'Westpac ATM Northcote', 'languageCode': 'en'},
  'reviews': [{'name': 'places/ChIJScpuOXxD1moRzGy1Mg-dGjw/reviews/Ci9DQUlRQUNvZENodHljRjlvT2tVNFFUTm9RVlZJZFRsTE5qWjVabEl0ZDFScE1HYxAB',
    'relativePublishTimeDescription': '2 months ago',
    'rating': 1,
    'text': {'text': 'Out of order. Not accessible. Appears to have been closed for well over a year.',
     'languageCode': 'en'},
    'originalText': {'text': 'Out of order. Not accessible. Appears to have been closed for well over a year.',
     'languageCode': 'en'},
    'authorAttribution': {'displayName': 'JayCee O',
     'uri': 'https://www.google.com/maps/contrib/112036275273933042679/reviews',
     'photoUri': 'https://lh3.googleusercontent.com/a/ACg8ocLJZv2kQZ-yhe0BPITxgVqXJAbRAkBN-ej3poUHhAS_cDgPtQ=s128-c0x00000

# Demo

In [135]:
OpenAI_client = OpenAI(api_key=openai_key)

# =======
# Input
# =======

User_Input = "Find me an ATM with 24-hour service along the way from CERES Organic Cafe to Melbourne TMJ & Facial Pain Centre."
user_prompt = "The geospatial query is: " + User_Input

# ============
# Feed to LLM 
# ============
response = client.chat.completions.create(
            model="gpt-4o-mini",  
            messages=[
                {"role": "system", "content": Geo_Decomp_system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.8
        )
        
# ===========================
# Extract & Parse LLM output
# ===========================
response_str = response.choices[0].message.content
llm_response = response_str
sub_queries = extract_stage2_json(llm_response)
print(sub_queries)

[{'id': 'q1', 'intent': 'geocodeLocation', 'entity': {'place': 'CERES Organic Cafe'}}, {'id': 'q2', 'intent': 'geocodeLocation', 'entity': {'place': 'Melbourne TMJ & Facial Pain Centre'}}, {'id': 'q3', 'intent': 'computeRoutes', 'entity': {'origin': '$q1', 'destination': '$q2'}}, {'id': 'q4', 'intent': 'searchAlongRoutes', 'entity': {'route': '$q3', 'POI_type': 'ATM', 'user_requirements': 'with 24-hour service'}}]


In [136]:
# ============
#  Call APIs
# ============

for sq in sub_queries:
    intent = sq["intent"]
    entity = resolve_placeholders(sq["entity"], results)

    if intent == "geocodeLocation":
        location = extract_location(entity)
        results[sq["id"]] = geocode_location(location)

    elif intent == "searchNearby":
        poi = normalize_and_validate_POI_type(entity["POI_type"])
        ref_loc = extract_reference_location(entity)
        radius = entity.get("radius", 10000)
        
        places_nearby = search_nearby(ref_loc, poi, radius)
        
        if "user_requirements" in entity:
            user_req = entity["user_requirements"]
            matched_places = semantic_matching(OpenAI_client, user_req, places_nearby)
            results[sq["id"]] = matched_places
        else:
            results[sq["id"]] = places_nearby
        
    elif intent == "computeRoutes":
        origin = entity["origin"]
        destination = entity["destination"]
        if "waypoint" in entity:
            waypoint = entity["waypoint"]
            route = compute_routes(origin, destination, waypoint=waypoint)
        else:
            route = compute_routes(origin, destination)
            
        results[sq["id"]] = route
        
    elif intent == "searchAlongRoutes":
        route = entity["route"]  # a list of route(s)
        textQuery = entity["POI_type"]
        
        places_along_routes = []
        for r in route:
            places = search_along_routes(route, textQuery)
            places_along_routes.extend(places)
        
        if "user_requirements" in entity:
            user_req = entity["user_requirements"]
            matched_places = semantic_matching(OpenAI_client, user_req, places_along_routes)
            results[sq["id"]] = matched_places
        else:
            results[sq["id"]] = places_along_routes

# results now holds the output for each sub-query by id
print(results)

{'q1': {'latitude': -37.7656718, 'longitude': 144.98295629999998}, 'q2': {'latitude': -37.872701, 'longitude': 145.0233725}, 'q3': [{'encodedPolyline': '~aoeFs|{sZHJ@F?\\PAjEf@bBNT?MyEHyAfAiN\\cGrB_YtFx@bZjD|@HlC^fEf@XAbBJlA[x@WxEeAb@Sj@q@dFmHl@m@lAiAJ]hAeUhO|A`ANz@\\hBfBVZjAjAVATCPM`@g@r@a@f@If@?~A`@jCd@vCZr@DTA`CLp@@nBLd@PvU`C^HHEtB`@pFj@rJr@bHn@fD`@lUxB~SnBdNrAvDZtMtAdKdAdBTfe@nEnIp@x@DlPbBlFf@~El@bCZjE^hA?hJeJfBqB~@_Bl@yAb@eB~@_GbB}L`AoH@aBIq@i@oDSoCYsBw@gDOyA@yAJ}@Tu@^q@rA{BTy@Fs@EWEmAOiC@w@L{@L[h@s@\\Mt@KX@LFzBTbBVpVhCzEp@T?lZlDhMzAxY|B~@DrUhCnLlAlAoP~@yN^kFd@uGVmDnB{Z\\aCZmA|Xqz@xAkEx@VpBXVJpS|B`QhB[lFf@F', 'origin': {'latitude': -37.7656718, 'longitude': 144.98295629999998}, 'destination': {'latitude': -37.872701, 'longitude': 145.0233725}, 'distanceMeters': 16179, 'duration': '2097s'}], 'q4': [{'id': 'ChIJScpuOXxD1moRzGy1Mg-dGjw', 'formattedAddress': '290 State Route 29, Northcote VIC 3070, Australia', 'location': {'latitude': -37.771682999999996, 'longitude': 144.9988339}, 'd

In [137]:
# ===============
# Parse Results
# ===============

final_subQuery_id = sub_queries[-1]["id"]  # last sub-query
final_subQuery_results = results[final_subQuery_id]

answers = []
for place in final_subQuery_results:
    answers.append({"displayName": place["displayName"]['text'],
                    "location": place["location"],
                    "id": place["id"],
                    "formattedAddress": place["formattedAddress"]
                    })

print(answers)

[{'displayName': 'Westpac ATM Northcote', 'location': {'latitude': -37.771682999999996, 'longitude': 144.9988339}, 'id': 'ChIJScpuOXxD1moRzGy1Mg-dGjw', 'formattedAddress': '290 State Route 29, Northcote VIC 3070, Australia'}, {'displayName': 'Precinct ATM Richmond Bridge Rd', 'location': {'latitude': -37.8189426, 'longitude': 145.00124499999998}, 'id': 'ChIJt_in5_ZC1moRbx7fHA3LtMY', 'formattedAddress': '300 Bridge Rd, Richmond VIC 3121, Australia'}, {'displayName': 'NAB ATM', 'location': {'latitude': -37.811571799999996, 'longitude': 144.9679682}, 'id': 'ChIJiTLwc8lC1moRX3wErc7vNps', 'formattedAddress': 'Shop 2-4/196-208 Russell St, Melbourne VIC 3000, Australia'}, {'displayName': 'Westpac ATM Brunswick', 'location': {'latitude': -37.7734545, 'longitude': 144.9611928}, 'id': 'ChIJbaFhSllD1moR3ozvSXqJGJI', 'formattedAddress': '192 Sydney Rd, Brunswick VIC 3056, Australia'}, {'displayName': 'Westpac ATM Fitzroy', 'location': {'latitude': -37.8026803, 'longitude': 144.9834285}, 'id': 'ChI

### An example of generalised case

In [ ]:
{
  "sub_queries": [
    {
      "id": "q1",
      "intent": "geocodeLocation",
      "entity": {
        "place": "Yarra River"
      }
    },
    {
      "id": "q2",
      "intent": "searchNearby",
      "entity": {
        "reference_location": "$q1",
        "POI_type": "school",
        "buffer_range": 500
      }
    },
    {
      "id": "q3",
      "intent": "searchNearby",
      "entity": {
        "reference_location": "$q2",
        "POI_type": "hospital",
        "buffer_range": 3000
      }
    }
  ]
}

# Experiments

In [9]:
import json
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
import requests
import re
import time
from openai import OpenAI
from collections import defaultdict
import json

from itertools import islice

import polyline
from shapely.geometry import LineString
from pyproj import Transformer
import matplotlib.pyplot as plt
import numpy as np

import json
import time
import traceback
import math
from statistics import mean
from geopy.distance import geodesic
import random

# ----------------
# Set up API Keys
# ----------------

load_dotenv()  # load .env file

openai_key = os.getenv("OPENAI_API_KEY")
google_maps_key = os.getenv("GOOGLE_MAPS_API_KEY")

# Build OpenAI_client
OpenAI_client = OpenAI(api_key=openai_key)

In [10]:
# =========================
#  Data Pre-processing
# =========================

# ### Read in benckmark dataset (in JSON form) ###
# with open("../finalQA_newVer.json", 'r') as f:
#     data_dict = json.load(f)
    
# ### Filter out queries with no Ground-truth ###  -- exclude those with no ground truth
# valid_data_dict = {key: val for key, val in data_dict.items() if val['answers']}        # 1846/1930 valid query


# ### sample 180 questions of different type (3 different types in total) ###
# # Assign question types
# def detect_question_type(query):
#     q = query.lower()
#     if "within" in q:
#         return 3
#     elif "passing" in q:
#         return 2
#     else:
#         return 1

# for key, val in valid_data_dict.items():
#     valid_data_dict[key]["question_type"] = detect_question_type(key)
    
# sample_per_type = 60
# type1_query = [key for key, val in valid_data_dict.items() if val["question_type"] == 1]  # In total 1158 type 1 query
# type2_query = [key for key, val in valid_data_dict.items() if val["question_type"] == 2]  # In total 221 type 2 query
# type3_query = [key for key, val in valid_data_dict.items() if val["question_type"] == 3]  # In total 551 type 3 query

# sampled_type1 = random.sample(type1_query, min(sample_per_type, len(type1_query)))
# sampled_type2 = random.sample(type2_query, min(sample_per_type, len(type2_query)))
# sampled_type3 = random.sample(type3_query, min(sample_per_type, len(type3_query)))

# test_query = sampled_type1 + sampled_type2 + sampled_type3
# test_data_dict = {key: val for key, val in valid_data_dict.items() if key in test_query}

# ### Save Experiment Data ###
# with open('../benchmark/test_QA.json', 'w') as f:
#     json.dump(test_data_dict, f, indent=4)

In [11]:
# ----------------
#  Helpers
# ----------------

def normalise_gt(gt):
    """
    Convert any ground-truth format into:
    [{
        "id": ..., (optional)
        "formattedAddress": ...,
        "location": {
            "latitude": ...,
            "longitude": ...
        },
        "displayName": ...
    }]
    """
    gt_list = []

    if not gt:
        return 'No Ground Truth'
    
    # For ground-truth of non buffer-range restricted query
    if isinstance(gt[0], dict) and "formattedAddress" not in gt[0].keys():
        for key, val in gt[0].items():
            gt_dict = dict()
            gt_dict['formattedAddress'] = key
            gt_dict['location'] = val['location']
            gt_dict['displayName'] = val['displayName']
            gt_list.append(gt_dict)
        return gt_list
    
    # others remain the same
    return gt

def haversine(coord1, coord2):
    """Compute distance (meters) between two (lat, lon) points."""
    R = 6371000  # radius of Earth in meters
    lat1, lon1 = coord1['latitude'], coord1['longitude']
    lat2, lon2 = coord2['latitude'], coord2['longitude']
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1-a))

def compute_topk_recall(system_results, ground_truth, k=5):
    """
    system_results: list of dicts from your system
    ground_truth:   list of dicts from ground truth
    k: number of top results to consider
    """
    if not system_results or not ground_truth:
        return None
    
    # Extract keys for matching: prefer id else formattedAddress
    if "id" in ground_truth[0].keys():
        sys_keys = {res.get("id") for res in system_results[:k]}
        gt_keys  = {gt.get("id") for gt in ground_truth[:k]}  # top-k ground truth
    else:
        sys_keys = {res.get("formattedAddress") for res in system_results[:k]}
        gt_keys  = {gt.get("formattedAddress") for gt in ground_truth[:k]}  # top-k ground truth

    overlap = sys_keys & gt_keys
    recall = len(overlap) / len(gt_keys) if gt_keys else 0.0
    return recall

def compute_avg_distance_deviation(system_results, ground_truth, k=5):
    """
    Compute average minimum distance (km) between each ground-truth and 
    its nearest system result in top-k. Order independent.
    """

    sys_topk = system_results[:k]
    gt_topk  = ground_truth[:k]

    if not sys_topk or not gt_topk:
        return None  # or 0.0

    distances = []
    for gt_item in gt_topk:
        gt_loc = (
            gt_item["location"]["latitude"],
            gt_item["location"]["longitude"]
        )
        # compute distance to each system item
        dists = [
            geodesic(
                gt_loc,
                (
                    sys_item["location"]["latitude"],
                    sys_item["location"]["longitude"]
                )
            ).meters
            for sys_item in sys_topk
        ]
        distances.append(min(dists))  # closest match

    return sum(distances) / len(distances) if distances else 0.0

In [12]:
# =========================
#  Core Evaluation Function
# =========================

def evaluate_query(input_query, ground_truth):
    """
    Run one query through the pipeline and compute metrics.
    ground_truth_ids: list of GT place IDs
    ground_truth_coords: list of (lat, lon) for GT places
    """
    record = {
        "user_input": input_query,
        "llm_response": None,
        "sub_queries": None,
        "final_results": None,
        "success": False,
        "runtime_sec": None,
        "top5_recall": None,
        "avg_dist_dev_m": None,
        "error": None
    }

    start_time = time.time()

    try:
        # --- 1. Geo-DecomP Prompting ---
        user_prompt = "The geospatial query is: " + input_query
        response = OpenAI_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": Geo_Decomp_system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.8
        )
        response_str = response.choices[0].message.content
        record["llm_response"] = response_str

        # Parse JSON (catch invalid JSON)
        sub_queries = extract_stage2_json(response_str)
        record["sub_queries"] = sub_queries

        # --- 2. Sub-query Orchestration ---
        results = {}
        for sq in sub_queries:
            try:
                intent = sq["intent"]
                entity = resolve_placeholders(sq["entity"], results)

                if intent == "geocodeLocation":
                    location = extract_location(entity)
                    results[sq["id"]] = geocode_location(location)

                elif intent == "searchNearby":
                    poi = normalize_and_validate_POI_type(entity["POI_type"])
                    ref_loc = extract_reference_location(entity)
                    radius = entity.get("radius", 10000)

                    ## Handle multiple reference locations ##
                    if isinstance(ref_loc, list):
                        places_nearby = []
                        for loc in ref_loc:
                            places_nearby.extend(search_nearby(loc, poi, radius))
                    else:
                        places_nearby = search_nearby(ref_loc, poi, radius)

                    ## Handle User Semantic Requirements ##
                    if "user_requirements" in entity:
                        user_req = entity["user_requirements"]
                        matched_places = semantic_matching(OpenAI_client, user_req, places_nearby)
                        results[sq["id"]] = matched_places
                    else:
                        results[sq["id"]] = places_nearby

                elif intent == "computeRoutes":
                    origin = entity["origin"]
                    destination = entity["destination"]
                    if "waypoint" in entity:
                        waypoint = entity["waypoint"]
                        route = compute_routes(origin, destination, waypoint=waypoint)
                    else:
                        route = compute_routes(origin, destination)

                    results[sq["id"]] = route

                elif intent == "searchAlongRoutes":
                    route = entity["route"]  # a list of route(s)
                    textQuery = entity["POI_type"]

                    places_along_routes = []
                    for r in route:
                        places = search_along_routes(route, textQuery)
                        places_along_routes.extend(places)

                    ## Handle User Semantic Requirements ##
                    if "user_requirements" in entity:
                        user_req = entity["user_requirements"]
                        matched_places = semantic_matching(OpenAI_client, user_req, places_along_routes)
                        results[sq["id"]] = matched_places
                    else:
                        results[sq["id"]] = places_along_routes

                elif intent == "sampleRoutePoints":
                    route = entity["route"]
                    num_route_points = entity["num_route_points"]
                    sampled_points = sample_route_points(route, n_points=num_route_points)
                    results[sq["id"]] = sampled_points

                else:
                    raise ValueError(f"Unknown intent: {intent}")

            except Exception as inner_e:
                traceback_str = traceback.format_exc()
                record["error"] = f"Error in subquery {sq.get('id')}: {inner_e}\n{traceback_str}"
                raise

        # --- Final Results ---
        final_subQuery_id = sub_queries[-1]["id"]
        final_subQuery_results = results[final_subQuery_id]
        answers = []
        for place in final_subQuery_results:
            answers.append({
                "displayName": place["displayName"]["text"],
                "location": place["location"],
                "id": place["id"],
                "formattedAddress": place["formattedAddress"]
            })

        record["final_results"] = answers
        record["success"] = True  # reached here with no exception

        # --- Metrics ---
        record["top5_recall"] = compute_topk_recall(answers, ground_truth, k=5)
        record["avg_dist_dev_m"] = compute_avg_distance_deviation(answers, ground_truth, k=5)

    except Exception as e:
        traceback_str = traceback.format_exc()
        record["error"] = f"{e}\n{traceback_str}"
        record["success"] = False

    end_time = time.time()
    record["runtime_sec"] = end_time - start_time

    return record

In [13]:
# =========================
#  Experiment Runner
# =========================

def run_experiments(data):
    """
    data: dict of query: data 
    """
    
    all_records = []
    for key, val in data.items():
        rec = evaluate_query(key, normalise_gt(val['answers']))
        all_records.append(rec)
        print(f"Done: {key}, success={rec['success']}, time={rec['runtime_sec']:.2f}s")
    
    return all_records

In [14]:
# ========== Run Experiments ==========

## Load Test data
with open('../benchmark/test_QA.json', 'r') as f:
    test_data_dict = json.load(f)

## run experiment
experiment_records = run_experiments(test_data_dict)

Done: Find me an ATM with no withdrawal fees along the way from Footscray Community Arts to South Kensington, passing through High Note., success=False, time=19.00s
Done: Find me a grocery store with extended opening hours along the way from Petty's Orchards to Deer Park Club, passing through Heavenly Queen Temple., success=True, time=91.13s
Done: Find me a bus station with free Wi-Fi along the way from One By One Asian Supermarket to Maribyrnong College, passing through Balaclava., success=True, time=20.81s
Done: Find me a restaurant with vegetarian options along the way from Dan Murphy's Harpley to Our Lady of Lebanon Church, within 6km of the route., success=True, time=33.06s
Done: Find me a restaurant that has a high rating on review sites along the way from Dan Murphy's Harpley to Our Lady of Lebanon Church, within 6km of the route., success=True, time=30.92s
Done: Find me a parking area that is open 24 hours along the way from El Jannah Chicken Kensington to Central Burgs, within

Done: Find me a bakery that offers vegan options along the way from Belgrave to The House of Mandi, passing through Shell Reddy Express Flemington., success=True, time=35.48s
Done: Find me a train station that has waiting areas along the way from The Pancake Parlour Malvern East to St John Bosco's School Niddrie., success=True, time=23.23s
Done: Find me a bus station that has amenities like restrooms along the way from Elster Elsternwick to Officeworks Frankston., success=False, time=9.52s
Done: Find me a grocery store with extended opening hours along the way from RACV Healesville Country Club & Resort to Pakenham Library - Myli - My Community Library., success=True, time=25.01s
Done: Find me a police station that is open 24 hours along the way from Woolworths Bentleigh to Green Refectory., success=True, time=21.95s
Done: Find me a beach with lifeguard services along the way from Carrum to Malvern Valley Public Golf Course., success=True, time=20.77s
Done: Find me a gas station that o

Done: Find me a parking area that is free along the way from Dinosaur World Somerville to KFC Caulfield, within 9km of the route., success=True, time=35.49s
Done: Find me a grocery store with a bakery section along the way from Sandringham Basketball Stadium to Bunnings Thomastown., success=True, time=21.66s
Done: Find me a tourist attraction with a rating of at least 4 stars along the way from KFC Footscray to Big W Melton, within 8km of the route., success=True, time=25.89s
Done: Find me a hotel with a fitness center along the way from Gold Leaf Eastern Restaurant to Spotlight Sandown., success=True, time=23.54s
Done: Find me a park that has children's play equipment along the way from Tru Coffee Balwyn to Katik Turkish Take Away, passing through The Croxton., success=True, time=32.69s
Done: Find me a bar with live music along the way from Bentleigh Gymbaroo to Nillumbik Estate., success=True, time=24.05s
Done: Find me a hospital with a trauma center along the way from Hungry Jack's 

Done: Find me a beach that allows dogs along the way from Coles Footscray to Stoneground Bakehouse., success=True, time=17.00s
Done: Find me a school that offers extracurricular activities along the way from Guzman y Gomez - Brunswick to Acme Binding Company, passing through Good Vibes Yoga., success=True, time=26.57s
Done: Find me a convenience store that is open 24 hours along the way from Kmart Templestowe to bp at Rose St, Coburg VIC 3058, Australia, passing through Roccella Italian Restaurant East Melbourne., success=False, time=19.70s
Done: Find me a hospital that has a pharmacy along the way from Woolworths Vermont to La Esquina Latina., success=True, time=20.21s
Done: Find me a hospital with specialized pediatric care along the way from Aberfeldie Park to Carnegie Library and Community Centre., success=True, time=23.73s
Done: Find me a bar that has a cozy atmosphere along the way from McDonald's at 280 Berwick-Cranbourne Rd, Clyde VIC 3978, Australia to Ten Minutes by Tractor.,

In [15]:
# ========== Save Experiments results ==========

# to_csv
df = pd.DataFrame(experiment_records)
df.to_csv("../experiments/ver1_expRes.csv", index=False)

# to JSON
with open("../experiments/ver1_expRes.json", "w") as f:
    json.dump(experiment_records, f, indent=4)

In [16]:
#pd.read_csv("../experiments/ver1_expRes.csv")

,user_input,llm_response,sub_queries,final_results,success,runtime_sec,top5_recall,avg_dist_dev_m,error
0,Find me an ATM with no withdrawal fees along t...,**Stage 1: Hierarchical (Top-down) Query Decom...,"[{'id': 'q1', 'intent': 'geocodeLocation', 'en...",NaN,False,18.995569,NaN,NaN,list index out of range\nTraceback (most recen...
1,Find me a grocery store with extended opening ...,**Stage 1: Hierarchical (Top-down) Query Decom...,"[{'id': 'q1', 'intent': 'geocodeLocation', 'en...","[{'displayName': 'FoodWorks', 'location': {'la...",True,91.130834,0.6,445.429844,NaN
2,Find me a bus station with free Wi-Fi along th...,**Stage 1: Hierarchical (Top-down) Query Decom...,"[{'id': 'q1', 'intent': 'geocodeLocation', 'en...",[{'displayName': 'Hoppers Crossing Station/Old...,True,20.809595,1.0,0.000000,NaN
3,Find me a restaurant with vegetarian options a...,**Stage 1: Hierarchical (Top-down) Query Decom...,"[{'id': 'q1', 'intent': 'geocodeLocation', 'en...","[{'displayName': 'DELHI NIGHTS - Truganina', '...",True,33.055848,0.6,955.790095,NaN
4,Find me a restaurant that has a high rating on...,**Stage 1: Hierarchical (Top-down) Query Decom...,"[{'id': 'q1', 'intent': 'geocodeLocation', 'en...",[{'displayName': 'DELHI NIGHTS - Hoppers Cross...,True,30.924334,0.4,1391.117193,NaN
...,...,...,...,...,...,...,...,...,...
175,Find me a gas station with convenience store a...,**Stage 1: Hierarchical (Top-down) Query Decom...,"[{'id': 'q1', 'intent': 'geocodeLocation', 'en...","[{'displayName': '7-Eleven', 'location': {'lat...",True,24.178913,0.6,1962.931377,NaN
176,Find me a supermarket that has a loyalty progr...,**Stage 1: Hierarchical (Top-down) Query Decom...,"[{'id': 'q1', 'intent': 'geocodeLocation', 'en...",NaN,False,27.602886,NaN,NaN,'NoneType' object is not subscriptable\nTraceb...
177,Find me a bank with an ATM service along the w...,**Stage 1: Hierarchical (Top-down) Query Decom...,"[{'id': 'q1', 'intent': 'geocodeLocation', 'en...",NaN,False,22.981861,NaN,NaN,'route'\nTraceback (most recent call last):\n ...
178,Find me a hospital with a high patient rating ...,**Stage 1: Hierarchical (Top-down) Query Decom...,"[{'id': 'q1', 'intent': 'geocodeLocation', 'en...","[{'displayName': 'Stanlake Private Hospital', ...",True,25.905469,0.8,1790.935002,NaN


In [ ]:
# with open("../experiments/ver1_expRes.json", "w") as f:
#     json.dump(experiment_records, f, indent=4)

## Baseline Performance 

In [34]:
# ===================
# Zero-shot LLM Baseline
# ===================

baseline_prompt = """
Return 5 results to the query. The result should be a JSON list, each JSON should have the following format:
    {
        "formattedAddress": string,
        "location": {
            "latitude": float,
            "longitude": float
        },
        "displayName": string
    }
"""

In [35]:
# =========================
#  Helper
# =========================

def extract_baseline_result(llm_answer):
    pattern = r'```json(.*?)```'
    match = re.search(pattern, llm_answer, re.DOTALL)

    if match:
        json_str = match.group(1).strip()
        try:
            parsed_json = json.loads(json_str)

        except json.JSONDecodeError as e:
            print("Failed to parse JSON:", e)
    else:
        print("No JSON block found")
        
    return parsed_json

In [36]:
# =========================
#  Baseline Evaluation Function
# =========================

def evaluate_baseline(input_query, ground_truth):
    """
    Run one query through the pipeline and compute metrics.
    ground_truth_ids: list of GT place IDs
    ground_truth_coords: list of (lat, lon) for GT places
    """
    record = {
        "user_input": input_query,
        "llm_response": None,
        "final_results": None,
        "success": False,
        "runtime_sec": None,
        "top5_recall": None,
        "avg_dist_dev_m": None,
        "error": None
    }

    start_time = time.time()

    try:
        # --- 1. Basline Prompting ---
        response = OpenAI_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": input_query + baseline_prompt}
            ],
            temperature=0.8
        )
        
        response_str = response.choices[0].message.content
        record["llm_response"] = response_str
        
        # extract answer
        answers = extract_baseline_result(response_str)

        record["final_results"] = answers
        record["success"] = True  # reached here with no exception

        # --- Metrics ---
        record["top5_recall"] = compute_topk_recall(answers, ground_truth, k=5)
        record["avg_dist_dev_m"] = compute_avg_distance_deviation(answers, ground_truth, k=5)

    except Exception as e:
        traceback_str = traceback.format_exc()
        record["error"] = f"{e}\n{traceback_str}"
        record["success"] = False

    end_time = time.time()
    record["runtime_sec"] = end_time - start_time

    return record

In [37]:
# =========================
#  Experiment Runner
# =========================

def run_baseline_experiments(data):
    """
    data: dict of query: data 
    """
    
    all_records = []
    for key, val in data.items():
        rec = evaluate_baseline(key, normalise_gt(val['answers']))
        all_records.append(rec)
        print(f"Done: {key}, success={rec['success']}, time={rec['runtime_sec']:.2f}s")
    
    return all_records

In [38]:
# ========== Run Experiments ==========

## Load Test data
with open('../benchmark/test_QA.json', 'r') as f:
    test_data_dict = json.load(f)

## run experiment
baseline_experiment_records = run_baseline_experiments(test_data_dict)

Done: Find me an ATM with no withdrawal fees along the way from Footscray Community Arts to South Kensington, passing through High Note., success=True, time=9.17s
Done: Find me a grocery store with extended opening hours along the way from Petty's Orchards to Deer Park Club, passing through Heavenly Queen Temple., success=True, time=9.85s
Done: Find me a bus station with free Wi-Fi along the way from One By One Asian Supermarket to Maribyrnong College, passing through Balaclava., success=True, time=12.31s
Done: Find me a restaurant with vegetarian options along the way from Dan Murphy's Harpley to Our Lady of Lebanon Church, within 6km of the route., success=True, time=10.74s
Done: Find me a restaurant that has a high rating on review sites along the way from Dan Murphy's Harpley to Our Lady of Lebanon Church, within 6km of the route., success=True, time=13.39s
Done: Find me a parking area that is open 24 hours along the way from El Jannah Chicken Kensington to Central Burgs, within 6 

Done: Find me a bakery that offers vegan options along the way from Belgrave to The House of Mandi, passing through Shell Reddy Express Flemington., success=True, time=7.37s
Done: Find me a train station that has waiting areas along the way from The Pancake Parlour Malvern East to St John Bosco's School Niddrie., success=True, time=6.76s
Done: Find me a bus station that has amenities like restrooms along the way from Elster Elsternwick to Officeworks Frankston., success=True, time=9.01s
Done: Find me a grocery store with extended opening hours along the way from RACV Healesville Country Club & Resort to Pakenham Library - Myli - My Community Library., success=True, time=9.22s
Done: Find me a police station that is open 24 hours along the way from Woolworths Bentleigh to Green Refectory., success=True, time=7.58s
Done: Find me a beach with lifeguard services along the way from Carrum to Malvern Valley Public Golf Course., success=True, time=7.84s
Done: Find me a gas station that offers 

Done: Find me a parking area that is free along the way from Dinosaur World Somerville to KFC Caulfield, within 9km of the route., success=True, time=8.87s
Done: Find me a grocery store with a bakery section along the way from Sandringham Basketball Stadium to Bunnings Thomastown., success=True, time=6.90s
Done: Find me a tourist attraction with a rating of at least 4 stars along the way from KFC Footscray to Big W Melton, within 8km of the route., success=True, time=7.68s
Done: Find me a hotel with a fitness center along the way from Gold Leaf Eastern Restaurant to Spotlight Sandown., success=True, time=5.94s
Done: Find me a park that has children's play equipment along the way from Tru Coffee Balwyn to Katik Turkish Take Away, passing through The Croxton., success=True, time=6.65s
Done: Find me a bar with live music along the way from Bentleigh Gymbaroo to Nillumbik Estate., success=True, time=8.81s
Done: Find me a hospital with a trauma center along the way from Hungry Jack's Burger

Done: Find me a beach that allows dogs along the way from Coles Footscray to Stoneground Bakehouse., success=True, time=7.68s
Done: Find me a school that offers extracurricular activities along the way from Guzman y Gomez - Brunswick to Acme Binding Company, passing through Good Vibes Yoga., success=True, time=9.42s
Done: Find me a convenience store that is open 24 hours along the way from Kmart Templestowe to bp at Rose St, Coburg VIC 3058, Australia, passing through Roccella Italian Restaurant East Melbourne., success=True, time=7.99s
Done: Find me a hospital that has a pharmacy along the way from Woolworths Vermont to La Esquina Latina., success=True, time=9.32s
Done: Find me a hospital with specialized pediatric care along the way from Aberfeldie Park to Carnegie Library and Community Centre., success=True, time=9.93s
Done: Find me a bar that has a cozy atmosphere along the way from McDonald's at 280 Berwick-Cranbourne Rd, Clyde VIC 3978, Australia to Ten Minutes by Tractor., succe

In [41]:
len(baseline_experiment_records)

180

In [40]:
# ========== Save Baseline Experiments results ==========

# to_csv
df_baseline = pd.DataFrame(baseline_experiment_records)
df_baseline.to_csv("../experiments/zeroShot_baseline_expRes.csv", index=False)

# to JSON
with open("../experiments/zeroShot_baseline_expRes.json", "w") as f:
    json.dump(baseline_experiment_records, f, indent=4)